# 第 36 课 · QLoRA：冻结四位基座上的低秩训练

**从“低比特推理能运行”走向“低比特基座上的训练能验证”。**

本课交付：本 Notebook 与同名 `.py`，两者都可独立使用；无需导入第 35 课的文件。

**完成后应能解释**：四位码为什么不更新但输入梯度仍然能传播；`prepare_model_for_kbit_training` 为什么在注入之前；初始 LoRA 应与哪一个基座比较；同一个 adapter 为何不能在未验证的量化方案之间随意切换。

> 验证边界：制作环境为 CPU PyTorch，未安装 Transformers/PEFT/bitsandbytes，真实 Qwen/CUDA 路径未执行。带输出的离线训练是真实执行；不是预训练模型能力、bitsandbytes内核或显存基准。真实入口默认关闭。

## 使用顺序

先运行初始化与折叠的工程定义，再从第 1 节学习。每次预测结果、运行核对、修改一个变量。练习默认不执行，避免一次“Run All”替你完成作业。

只需现有 PyTorch。真实依赖安装指令放在后文，**Notebook 不自动安装依赖、不自动下载权重**。代码单元 `Shift+Enter` 从上到下执行；第一次可只读原理和机制实验，完整离线 SFT 有独立开关。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第36课：QLoRA——冻结四位基座上的低秩训练。
默认：纯PyTorch教学NF4码本、打包与SFT，整层反量化FP32；不是bitsandbytes内核。
--run：真实Qwen3-0.6B + bitsandbytes NF4/双重量化 + PEFT；显式CUDA，不回退。
--inspect：只加载配套分词器。--reload-run：按同一量化契约恢复，不合并权重。
不依赖旧课程文件。默认只用PyTorch；真实路径显式运行才需要其他依赖与权重。
教学文件格式不是bitsandbytes格式；离线未实现双重量化或分页优化器。
只保存推理所需基座契约与adapter，不保存完整断点续训状态。
来源与版本见同名Notebook；课程固定版本不是最新版声明。
"""
from __future__ import annotations
import argparse
import contextlib
import copy
import gc
import hashlib
import importlib.metadata
import json
import math
import random
import re
import sys
import traceback
import tempfile
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace
from typing import Callable

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

MODEL_ID = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
SEED = 35  # 与第35课固定数据划分相同；不是从报告拷贝成绩
IGNORE = -100
FORMAT = "lesson36-qlora-v1"

### 共用工程定义：来自已学过的数据和训练流程

下面五个折叠单元把数据约定、样本切分、教学网络、token 加权损失和评分流程直接内置。**不要跳过执行**，但本课无需重新抄写这些已学内容。可展开阅读，函数与完整脚本一致。它们只定义函数，不训练、不联网。

本课仍保留 assistant-only 同位标签、内部移位禁止重复、PAD 与结束符分离处理、按来源 train/dev/test 划分和缺失记录固定分母。

In [2]:
# %% 01. 继承第34课已经检查的数据契约；代码内置，不依赖旧文件。
def require(condition, message: str) -> None:
    if not bool(condition):
        raise ValueError(message)

def digest(value) -> str:
    text = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def write_jsonl(path: Path, rows) -> None:
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")

def unique_object(pairs):
    out = {}
    for key, value in pairs:
        require(key not in out, f"JSON重复键：{key}")
        out[key] = value
    return out

def read_jsonl(path) -> list[dict]:
    records = []
    with Path(path).open(encoding="utf-8") as f:
        for number, line in enumerate(f, 1):
            require(bool(line.strip()), f"第{number}行为空；每行必须是一条JSON记录")
            try:
                records.append(json.loads(line, object_pairs_hook=unique_object))
            except (ValueError, json.JSONDecodeError) as exc:
                raise ValueError(f"第{number}行：{exc}") from exc
    audit_records(records)
    return records

def validate_record(record: dict) -> None:
    require(isinstance(record, dict), "记录必须是字典")
    require(set(record) == {"id", "group_id", "messages"}, "字段必须恰好为id/group_id/messages")
    for field in ("id", "group_id"):
        require(isinstance(record[field], str) and bool(record[field].strip()), f"{field}必须非空")
    messages = record["messages"]
    require(isinstance(messages, list) and len(messages) >= 2, "至少需要一轮完整问答")
    for i, message in enumerate(messages):
        require(isinstance(message, dict) and set(message) == {"role", "content"}, f"消息{i}字段错误")
        require(message["role"] in ("system", "user", "assistant"), f"消息{i}角色不支持")
        require(isinstance(message["content"], str) and bool(message["content"].strip()), f"消息{i}内容不能为空")
        # 验证可编码，但不strip/归一化真实内容，保留代码缩进和换行。
        try:
            message["content"].encode("utf-8", errors="strict")
        except UnicodeError as exc:
            raise ValueError(f"消息{i}不是有效UTF-8文本") from exc
    start = int(messages[0]["role"] == "system")
    require(len(messages[start:]) >= 2 and len(messages[start:]) % 2 == 0, "对话必须结束于完整assistant回答")
    for i, message in enumerate(messages[start:]):
        expected = "user" if i % 2 == 0 else "assistant"
        require(message["role"] == expected, f"第{start+i}条应为{expected}")

def audit_records(records: list[dict]) -> None:
    require(isinstance(records, list) and bool(records), "数据集不能为空")
    ids, contents = set(), set()
    for record in records:
        validate_record(record)
        require(record["id"] not in ids, f"重复id：{record['id']}")
        signature = digest(record["messages"])
        require(signature not in contents, "发现逐字重复对话：先人工确认来源并显式去重")
        ids.add(record["id"])
        contents.add(signature)

def prompt_keys(record: dict) -> set[str]:
    """每个assistant答案之前的完整消息前缀。仅发现精确重复，不做语义去重。"""
    return {digest(record["messages"][:i]) for i, m in enumerate(record["messages"])
            if m["role"] == "assistant"}

def labels_from_mask(ids: list[int], mask: list[int]) -> list[int]:
    require(isinstance(ids, list) and len(ids) >= 2, "input_ids至少两个位置")
    require(all(type(x) is int and x >= 0 for x in ids), "token id必须为非负整数，排除bool")
    require(isinstance(mask, list) and len(mask) == len(ids), "mask长度必须等于输入长度")
    require(all(type(x) in (bool, int) and x in (0, 1) for x in mask), "mask只能是0/1")
    require(not mask[0], "首位置没有前驱logit，不能成为监督目标")
    require(any(mask[1:]), "移位后无有效监督token；拒绝全忽略样本")
    return [token if chosen else IGNORE for token, chosen in zip(ids, mask)]

def select_mask(length: int, spans: list[dict], policy="all") -> list[int]:
    require(policy in ("all", "last"), "policy应为all/last")
    assistant = [s for s in spans if s["role"] == "assistant"]
    require(bool(assistant), "缺少assistant区间")
    chosen = assistant if policy == "all" else assistant[-1:]
    result = [0] * length
    for span in chosen:
        begin, end = span["target_start"], span["target_end"]
        require(0 < begin < end <= length, "监督区间非法")
        for p in range(begin, end):
            result[p] = 1
    return result

def make_row(record, ids, spans, policy, codec_contract) -> dict:
    mask = select_mask(len(ids), spans, policy)
    labels = labels_from_mask(ids, mask)
    return {"id":record["id"], "group_id":record["group_id"], "input_ids":ids,
            "attention_mask":[1]*len(ids), "labels":labels,
            "assistant_mask":mask, "spans":spans, "policy":policy,
            "codec_fingerprint":digest(codec_contract), "source_fingerprint":digest(record["messages"])}

class ToyChatCodec:
    """显式role token + UTF-8字节 + EOT。故意令PAD==EOT以检验按长度补齐。"""
    roles = {"system":256, "user":257, "assistant":258}
    eot_id = 259
    pad_id = 259
    vocab_size = 260

    def contract(self) -> dict:
        return {"kind":"teaching-byte-codec-not-Qwen", "version":1, "roles":self.roles,
                "eot_id":self.eot_id, "pad_id":self.pad_id, "vocab_size":self.vocab_size,
                "targets":"selected assistant UTF-8 body + EOT; headers excluded"}

    def encode(self, record, policy="all") -> dict:
        validate_record(record)
        ids, spans = [], []
        for i, message in enumerate(record["messages"]):
            header = len(ids)
            ids.append(self.roles[message["role"]])
            begin = len(ids)
            ids.extend(message["content"].encode("utf-8"))
            ids.append(self.eot_id)
            spans.append({"message_index":i, "role":message["role"], "header_start":header,
                          "target_start":begin, "target_end":len(ids), "eot_index":len(ids)-1})
        return make_row(record, ids, spans, policy, self.contract())

    def display(self, ids):
        reverse = {v:k for k,v in self.roles.items()}
        pieces, buf = [], bytearray()
        def flush():
            if buf:
                pieces.append(bytes(buf).decode("utf-8", errors="backslashreplace")); buf.clear()
        for token in ids:
            if 0 <= token <= 255:
                buf.append(token)
            else:
                flush(); pieces.append("<EOT>" if token == self.eot_id else f"<{reverse.get(token, '?')}>")
        flush()
        return "".join(pieces)

def validate_row(row):
    require(isinstance(row, dict), "样本必须是字典")
    for key in ("input_ids", "labels", "attention_mask"):
        require(key in row and isinstance(row[key], list), f"缺少列表字段{key}")
    n = len(row["input_ids"])
    require(n >= 2 and len(row["labels"]) == n and len(row["attention_mask"]) == n, "长度不一致")
    require(all(type(x) is int and x >= 0 for x in row["input_ids"]), "非法input_ids")
    require(row["attention_mask"] == [1]*n, "collator接收未padding样本")
    require(all(type(y) is int and (y == IGNORE or y == x) for x,y in zip(row["input_ids"],row["labels"])),
            "labels必须与input_ids同位，不能预先移位")
    require(row["labels"][0] == IGNORE and any(y != IGNORE for y in row["labels"][1:]), "有效目标为空或首位被监督")

def collate_rows(rows, pad_id: int, multiple=1):
    require(isinstance(rows, list) and bool(rows), "不能整理空批次")
    require(type(pad_id) is int and pad_id >= 0, "pad_id必须为非负整数")
    require(type(multiple) is int and multiple >= 1, "multiple必须为正整数")
    for row in rows:
        validate_row(row)
    width = max(len(r["input_ids"]) for r in rows)
    width = math.ceil(width/multiple)*multiple
    batch = {"input_ids":[], "attention_mask":[], "labels":[]}
    for row in rows:
        n = len(row["input_ids"])
        batch["input_ids"].append(row["input_ids"][:] + [pad_id]*(width-n))
        batch["attention_mask"].append([1]*n + [0]*(width-n))
        batch["labels"].append(row["labels"][:] + [IGNORE]*(width-n))
    # 只返回模型接受的三个字段，不把id/spans/来源等传给model(**batch)。
    return batch

def qwen_char_spans(text: str, messages: list[dict]) -> list[dict]:
    """只用于本课固定Qwen纯文本模板。遇到未知格式明确失败，不猜边界。"""
    start_tag, end_tag = "<|im_start|>", "<|im_end|>"
    pos, spans = 0, []
    empty_think = "<think>\n\n</think>\n\n"
    for i, message in enumerate(messages):
        header = start_tag + message["role"] + "\n"
        require(text.startswith(header, pos), f"第{i}条消息模板头部不匹配")
        begin = pos + len(header)
        end = text.find(end_tag, begin)
        require(end >= 0, "模板缺少消息结束符")
        body = text[begin:end]
        require(start_tag not in body, "消息体包含控制分隔符")
        allowed = {message["content"]}
        if message["role"] == "assistant":
            # 固定Qwen模板可能给最后assistant自动加入空thinking区。
            allowed.add(empty_think + message["content"].lstrip("\n"))
        require(body in allowed, "序列化正文发生未支持的变换：不猜测、不中途替换模板")
        target_end = end + len(end_tag)
        spans.append({"message_index":i, "role":message["role"], "char_start":begin,
                      "char_end":target_end, "body":body})
        require(text.startswith("\n", target_end), "结束符后缺少预期换行")
        pos = target_end + 1
    require(pos == len(text), "模板有多余尾部；训练序列不可加下一assistant前缀")
    return spans

def offsets_to_token_spans(offsets, char_spans, ids, eot_id):
    """边界跨token时拒绝，避免悄悄把header/user监督进来。"""
    require(len(offsets) == len(ids), "offset长度错误")
    token_spans = []
    for span in char_spans:
        start, end = span["char_start"], span["char_end"]
        included = []
        for j, (a,b) in enumerate(offsets):
            if b <= a:
                continue
            if a < end and b > start:
                require(a >= start and b <= end, "token跨越监督边界；本课拒绝这种分词布局")
                included.append(j)
        require(bool(included), "模板区间没有映射到token")
        require(included == list(range(included[0], included[-1]+1)), "监督区间不连续")
        require(ids[included[-1]] == eot_id, "结束token不在监督区间内")
        token_spans.append({"message_index":span["message_index"], "role":span["role"],
                            "target_start":included[0], "target_end":included[-1]+1,
                            "eot_index":included[-1]})
    return token_spans

class QwenChatCodec:
    """真实tokenizer专用适配器。只加载tokenizer；不加载Qwen权重，也不依赖PEFT。"""
    def __init__(self, local_files_only=False):
        try:
            from transformers import AutoTokenizer
        except ImportError as exc:
            raise RuntimeError("真实分词器路径需要transformers；离线部分不需要。") from exc
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, use_fast=True,
                                                       trust_remote_code=False, local_files_only=local_files_only)
        require(self.tokenizer.is_fast, "本课需要fast tokenizer的offset_mapping")
        require(isinstance(self.tokenizer.chat_template, str), "模板必须为明确字符串")
        self.eot_id = self.tokenizer.convert_tokens_to_ids("<|im_end|>")
        require(type(self.eot_id) is int and self.tokenizer.encode("<|im_end|>", add_special_tokens=False)==[self.eot_id],
                "结束符必须是单独的已知特殊token")
        self.pad_id = self.tokenizer.pad_token_id
        require(type(self.pad_id) is int, "缺少pad_id；本课不自动改写tokenizer")
        self._contract = {"kind":"qwen-fixed-revision-real-tokenizer", "model_id":MODEL_ID, "revision":REVISION,
                          "template_sha256":digest(self.tokenizer.chat_template),
                          "vocab_sha256":digest(self.tokenizer.get_vocab()), "pad_id":self.pad_id, "eot_id":self.eot_id,
                          "enable_thinking":False, "add_generation_prompt":False,
                          "targets":"selected serialized assistant body (including template empty think if present) + im_end; header/trailing newline excluded",
                          "transformers":importlib.metadata.version("transformers")}

    def contract(self):
        return copy.deepcopy(self._contract)

    def encode(self, record, policy="all"):
        validate_record(record)
        tok = self.tokenizer
        forbidden = set(tok.all_special_tokens) | {"<think>", "</think>"}
        for m in record["messages"]:
            require(not any(s and s in m["content"] for s in forbidden),
                    "本课真实路径不支持正文中的保留控制串/显式thinking；不会靠扫描正文猜角色")
        kwargs = dict(add_generation_prompt=False, enable_thinking=False)
        text = tok.apply_chat_template(record["messages"], tokenize=False, **kwargs)
        full = tok.apply_chat_template(record["messages"], tokenize=True, **kwargs)
        enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
        require(enc["input_ids"] == full, "直接模板与文本再编码的ID不一致")
        chars = qwen_char_spans(text, record["messages"])
        spans = offsets_to_token_spans(enc["offset_mapping"], chars, full, self.eot_id)
        row = make_row(record, full, spans, policy, self.contract())
        row["rendered_text"] = text
        row["character_spans"] = chars
        return row

    def display(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)

    def native_mask_audit(self, record):
        """只探测，不把未验证的native mask作为训练标签。"""
        template = self.tokenizer.chat_template
        if not re.search(r"\{%[-+]?\s*generation\b", template):
            return {"status":"unsupported_template", "reason":"未发现generation标记；不能把全零mask当成功"}
        try:
            value = self.tokenizer.apply_chat_template(record["messages"], tokenize=True, return_dict=True,
                        return_assistant_tokens_mask=True, add_generation_prompt=False, enable_thinking=False)
            expected = self.encode(record, "all")
            mask = value.get("assistant_masks")
            require(isinstance(mask, list) and len(mask)==len(expected["input_ids"]), "native mask缺失或长度错误")
            require(value["input_ids"] == expected["input_ids"] and any(mask), "native ID错误或mask全零")
            same = mask == expected["assistant_mask"]
            return {"status":"same_contract" if same else "different_contract_do_not_substitute",
                    "native_count":sum(mask), "our_count":sum(expected["assistant_mask"])}
        except Exception as exc:
            return {"status":"probe_failed_not_used", "error":f"{type(exc).__name__}: {exc}"}

In [3]:
# %% 02. 三路数据划分与严格任务协议

def environment():
    versions = {}
    for name in ('torch', 'transformers', 'peft', 'accelerate', 'bitsandbytes', 'safetensors'):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return {'python':sys.version.split()[0], 'versions':versions,
            'cuda_available':torch.cuda.is_available(),
            'cuda_device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}


def new_run(root, kind):
    directory = Path(root).resolve() / (kind + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    directory.mkdir(parents=True, exist_ok=False)
    return directory


def quantity_object(text):
    """严格任务：一个JSON对象，恰好quantity键，非bool整数0..9999。不修复模型回答。"""
    require(isinstance(text, str), '回答必须是字符串')
    def nonfinite(value):
        raise ValueError('禁止NaN/Infinity：' + value)
    obj = json.loads(text, object_pairs_hook=unique_object, parse_constant=nonfinite)
    require(type(obj) is dict and set(obj) == {'quantity'}, '必须恰好含quantity字段')
    require(type(obj['quantity']) is int and 0 <= obj['quantity'] <= 9999, 'quantity必须是0..9999的整数，排除bool')
    return obj


def make_records():
    """24个来源、每个来源两种末轮问法。英文短文本方便字节教学；不是公开基准。"""
    records=[]
    for g in range(24):
        qty=g % 4 + 1
        item=['tea','rice','pen'][g % 3]
        for v, question in enumerate(['Return the quantity as JSON.', 'Only the quantity field, please.']):
            records.append({'id':f'R{g:02d}-{v}', 'group_id':f'order-R{g:02d}', 'messages':[
                {'role':'system','content':'Read the stock record. Return only JSON with integer key quantity.'},
                {'role':'user','content':f'Order R{g:02d}: {item}; quantity={qty}.'},
                {'role':'assistant','content':f'Noted order R{g:02d}.'},
                {'role':'user','content':question},
                {'role':'assistant','content':json.dumps({'quantity':qty},separators=(',',':'))},
            ]})
    return records


def split_three(records, seed=SEED):
    audit_records(records)
    for record in records:
        quantity_object(record['messages'][-1]['content'])
    groups=sorted({r['group_id'] for r in records})
    require(len(groups)>=6, '至少六个独立来源；不足时不要拆开同一来源凑数量')
    random.Random(seed).shuffle(groups)
    n=max(1,len(groups)//6)
    mapping={g:('test' if i<n else 'dev' if i<2*n else 'train') for i,g in enumerate(groups)}
    result={part:[copy.deepcopy(r) for r in records if mapping[r['group_id']]==part]
            for part in ('train','dev','test')}
    validate_splits(result)
    return result


def validate_splits(parts):
    require(set(parts)=={'train','dev','test'}, '必须恰好有train/dev/test')
    audit_records([r for values in parts.values() for r in values])
    seen_groups,seen_prompts=set(),set()
    for name, rows in parts.items():
        require(bool(rows),name+'不能为空')
        groups={r['group_id'] for r in rows}
        prompts=set().union(*(prompt_keys(r) for r in rows))
        require(groups.isdisjoint(seen_groups), '来源跨split，存在泄漏')
        require(prompts.isdisjoint(seen_prompts), '精确答案前缀跨split；检查来源关系')
        seen_groups.update(groups);seen_prompts.update(prompts)
    return {name:{'records':len(rows),'groups':len({r['group_id'] for r in rows})}
            for name,rows in parts.items()}


def public_messages(record):
    validate_record(record)
    return copy.deepcopy(record['messages'][:-1])


def prompt_ids(codec, record):
    messages=public_messages(record)
    if isinstance(codec, ToyChatCodec):
        ids=[]
        for m in messages:
            ids += [codec.roles[m['role']]] + list(m['content'].encode('utf-8')) + [codec.eot_id]
        return ids + [codec.roles['assistant']]
    return codec.tokenizer.apply_chat_template(messages, tokenize=True,
        add_generation_prompt=True, enable_thinking=False)


def prepare_splits(parts, codec, max_length):
    """不默默丢弃测试题。任意超长或非法样本都会停止，调整预算后重开实验。"""
    validate_splits(parts)
    result={}
    for part, records in parts.items():
        result[part]=[]
        for record in records:
            row=codec.encode(record, policy='last')
            validate_row(row)
            require(len(row['input_ids'])<=max_length,
                    f"{record['id']}长度{len(row['input_ids'])}超过{max_length}；不截断答案、不删除题目")
            require(len(prompt_ids(codec,record))<max_length, '生成提示超过约定预算')
            row['split']=part
            result[part].append(row)
    return result


@dataclass
class Settings:
    output_dir: str = 'lesson_36_outputs'
    device: str = 'cpu'
    dtype: str = 'float32'
    epochs: int = 8
    micro_batch: int = 2
    accumulation: int = 4
    lr: float = 0.01
    rank: int = 8
    alpha: float = 16.0
    max_length: int = 256
    max_new_tokens: int = 32
    seed: int = SEED
    gradient_checkpointing: bool = False
    local_files_only: bool = False
    targets: str = "qv"
    block_size: int = 64


def validate_settings(s):
    for name in ('epochs','micro_batch','accumulation','rank','max_length','max_new_tokens'):
        require(type(getattr(s,name)) is int and getattr(s,name)>0,name+'必须为正整数')
    require(math.isfinite(s.lr) and s.lr>0,'lr须有限正数')
    require(math.isfinite(s.alpha) and s.alpha>0,'alpha须有限正数')
    require(s.device in ('cpu','cuda') and s.dtype in ('float32','float16','bfloat16'), '设备/精度不支持')
    require(s.device!='cpu' or s.dtype=='float32','本课CPU统一FP32')
    require(s.targets in ('qv','all-linear'), 'targets应为qv/all-linear')
    require(type(s.block_size) is int and s.block_size > 0, 'block_size必须正整数')


def runtime(s):
    validate_settings(s)
    if s.device=='cuda':
        require(torch.cuda.is_available(),'显式CUDA请求失败；不回退CPU')
        if s.dtype=='bfloat16':
            require(torch.cuda.is_bf16_supported(),'本环境未声明BF16可用')
    return torch.device(s.device), getattr(torch,s.dtype)


def autocast(device,dtype):
    return torch.autocast(device_type=device.type,dtype=dtype) if device.type=='cuda' and dtype!=torch.float32 else contextlib.nullcontext()

In [4]:
# %% 03. 离线微型因果网络及手写LoRA：明确不是PEFT或预训练模型
class ToyAttention(nn.Module):
    def __init__(self,d=32,h=4):
        super().__init__();self.d=d;self.h=h
        self.q_proj=nn.Linear(d,d,bias=False);self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False);self.o_proj=nn.Linear(d,d,bias=False)
    def forward(self,x,mask):
        b,t,d=x.shape
        q,k,v=[p(x).reshape(b,t,self.h,d//self.h).transpose(1,2) for p in (self.q_proj,self.k_proj,self.v_proj)]
        allowed=torch.ones(t,t,dtype=torch.bool,device=x.device).tril()[None,None]&mask[:,None,None,:].bool()
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=allowed,dropout_p=0.0)
        return self.o_proj(y.transpose(1,2).reshape(b,t,d))

class ToyBlock(nn.Module):
    def __init__(self,d=32):
        super().__init__();self.norm1=nn.LayerNorm(d);self.norm2=nn.LayerNorm(d)
        self.self_attn=ToyAttention(d);self.ffn=nn.Sequential(nn.Linear(d,d*2),nn.GELU(),nn.Linear(d*2,d))
    def forward(self,x,mask):
        x=x+self.self_attn(self.norm1(x),mask)
        return x+self.ffn(self.norm2(x))

class ToyLM(nn.Module):
    def __init__(self,max_length=256):
        super().__init__();self.config=SimpleNamespace(use_cache=False,max_position_embeddings=max_length)
        self.embed=nn.Embedding(260,32);self.pos=nn.Embedding(max_length,32)
        self.blocks=nn.ModuleList([ToyBlock(),ToyBlock()]);self.norm=nn.LayerNorm(32)
        self.head=nn.Linear(32,260,bias=False);self.checkpointed=False
    def forward(self,input_ids,attention_mask,**kwargs):
        require(input_ids.shape[1]<=self.config.max_position_embeddings,'超出教学模型位置表')
        x=self.embed(input_ids)+self.pos(torch.arange(input_ids.shape[1],device=input_ids.device))
        for block in self.blocks:
            x=checkpoint(block,x,attention_mask,use_reentrant=False) if self.training and self.checkpointed else block(x,attention_mask)
        return SimpleNamespace(logits=self.head(self.norm(x)))

class ToyLoRALinear(nn.Module):
    def __init__(self,base,rank,alpha):
        super().__init__();self.base_layer=base
        self.lora_A=nn.Parameter(torch.empty(rank,base.in_features))
        self.lora_B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5));self.scale=alpha/rank;self.enabled=True
    def forward(self,x):
        y=self.base_layer(x)
        return y+self.scale*F.linear(F.linear(x,self.lora_A),self.lora_B) if self.enabled else y

class ToyAdapter(nn.Module):
    def __init__(self,base,rank,alpha):
        super().__init__();self.base=base;base.requires_grad_(False)
        for block in base.blocks:
            for name in ('q_proj','v_proj'):
                setattr(block.self_attn,name,ToyLoRALinear(getattr(block.self_attn,name),rank,alpha))
    def forward(self,**kwargs):return self.base(**kwargs)
    @contextlib.contextmanager
    def disable_adapter(self):
        layers=[m for m in self.modules() if isinstance(m,ToyLoRALinear)];states=[m.enabled for m in layers]
        for m in layers:m.enabled=False
        try:yield
        finally:
            for m,state in zip(layers,states):m.enabled=state


def adapter_parameters(model):
    pairs=[(n,p) for n,p in model.named_parameters() if p.requires_grad]
    require(bool(pairs) and all('lora_A' in n or 'lora_B' in n for n,p in pairs), '存在非adapter可训练参数或全部冻结')
    return pairs


def assert_optimizer(model,optimizer):
    expected={id(p) for _,p in adapter_parameters(model)}
    actual=[p for group in optimizer.param_groups for p in group['params']]
    require(len(actual)==len({id(p) for p in actual}) and {id(p) for p in actual}==expected,'优化器未精确匹配当前adapter')


def snapshot_adapter(model):
    return {n:p.detach().cpu().clone() for n,p in adapter_parameters(model)}


def restore_adapter(model,state):
    params=dict(adapter_parameters(model));require(params.keys()==state.keys(),'adapter键集合不一致')
    with torch.no_grad():
        for name,p in params.items():
            require(p.shape==state[name].shape and bool(torch.isfinite(state[name]).all()), 'adapter形状/数值无效')
            p.copy_(state[name].to(device=p.device,dtype=p.dtype))


def hash_parameters(pairs):
    h=hashlib.sha256()
    for name,p in pairs:
        h.update(f'{name}|{list(p.shape)}|{p.dtype}'.encode())
        flat=p.detach().reshape(-1)
        for start in range(0,flat.numel(),1<<20):
            h.update(flat[start:start+(1<<20)].cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [5]:
# %% 04. 同位标签的损失、有效token加权、窗口边界更新

def tensor_batch(rows,pad_id,device):
    return {k:torch.tensor(v,dtype=torch.long,device=device) for k,v in collate_rows(rows,pad_id).items()}


def loss_parts(logits,labels):
    require(logits.ndim==3 and labels.ndim==2 and logits.shape[:2]==labels.shape,'logits/labels形状错误')
    require(labels.dtype==torch.long,'标签须为long')
    # 外部只移位一次，forward没有传labels，不使用库内loss。
    target=labels[:,1:].contiguous();count=int((target!=IGNORE).sum().item())
    require(count>0,'无有效next-token监督')
    loss_sum=F.cross_entropy(logits[:,:-1].float().reshape(-1,logits.shape[-1]),target.reshape(-1),ignore_index=IGNORE,reduction='sum')
    require(bool(torch.isfinite(loss_sum)), 'loss非有限，停止而不是删除此样本')
    return loss_sum,count


def forward_loss(model,batch,device,dtype):
    with autocast(device,dtype):
        logits=model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask'],use_cache=False).logits
        return loss_parts(logits,batch['labels'])


def windows(rows,micro_batch,accumulation,seed):
    require(type(micro_batch) is int and micro_batch>0 and type(accumulation) is int and accumulation>0,'批次参数必须正整数')
    order=list(range(len(rows)));random.Random(seed).shuffle(order)
    micro=[[rows[j] for j in order[i:i+micro_batch]] for i in range(0,len(order),micro_batch)]
    return [micro[i:i+accumulation] for i in range(0,len(micro),accumulation)]


def window_targets(window):
    return sum(sum(y!=IGNORE for y in r['labels'][1:]) for batch in window for r in batch)


def update_window(model,window,optimizer,scaler,device,dtype,pad_id):
    assert_optimizer(model,optimizer)
    n=window_targets(window);require(n>0,'窗口监督目标为空')
    model.train();optimizer.zero_grad(set_to_none=True)
    total=0.0
    for rows in window:
        batch=tensor_batch(rows,pad_id,device)
        summed,count=forward_loss(model,batch,device,dtype)
        total+=float(summed.detach())
        scaler.scale(summed/n).backward()
        del summed, batch
    scaler.unscale_(optimizer)
    params=[p for _,p in adapter_parameters(model)]
    finite=all(p.grad is None or bool(torch.isfinite(p.grad).all()) for p in params)
    require(any(p.grad is not None for p in params),'全部adapter都没有梯度')
    if not scaler.is_enabled():require(finite,'未启用scaler时出现非有限梯度')
    norm=nn.utils.clip_grad_norm_(params,1.0,error_if_nonfinite=True) if finite else torch.tensor(float('inf'))
    calls=[]
    hook=optimizer.register_step_post_hook(lambda *args: calls.append(True))
    try:
        scale_before=float(scaler.get_scale());scaler.step(optimizer);scaler.update()
    finally:hook.remove()
    optimizer.zero_grad(set_to_none=True)
    return {'targets':n,'mean_loss':total/n,'micro_batches':len(window),
            'ids':[r['id'] for batch in window for r in batch], 'updated':len(calls)==1,
            'grad_norm_before_clip':float(norm) if torch.isfinite(norm) else None,
            'scale_before':scale_before,'scale_after':float(scaler.get_scale())}


def evaluate_loss(model,rows,pad_id,device,dtype,batch_size=1):
    require(bool(rows),'评测集为空');require(batch_size>0,'batch_size必须正数')
    previous=model.training;model.eval();total=0.0;n=0
    try:
        with torch.no_grad():
            for i in range(0,len(rows),batch_size):
                summed,count=forward_loss(model,tensor_batch(rows[i:i+batch_size],pad_id,device),device,dtype)
                total+=float(summed);n+=count
    finally:model.train(previous)
    return {'loss':total/n,'loss_sum':total,'targets':n,'records':len(rows)}


class DevSelector:
    """只接收dev指标；最佳权重必须clone，不保存随训练变化的引用。"""
    def __init__(self):self.loss=math.inf;self.epoch=None;self.state=None
    def consider(self,split,epoch,loss,model):
        require(split=='dev','禁止用test/train指标选择adapter')
        require(math.isfinite(loss) and loss>=0,'dev loss无效')
        if loss<self.loss:
            self.loss=loss;self.epoch=epoch;self.state=snapshot_adapter(model);return True
        return False


def fit(model,rows,s,pad_id,device,dtype,directory):
    require(set(rows)=={'train','dev'},'fit只能接收train/dev，不能传test')
    train_params=[p for _,p in adapter_parameters(model)]
    optimizer=torch.optim.AdamW(train_params,lr=s.lr,weight_decay=0.0)
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda' and dtype==torch.float16)
    selector=DevSelector();history=[];attempted=effective=0
    initial=evaluate_loss(model,rows['dev'],pad_id,device,dtype)
    selector.consider('dev',0,initial['loss'],model)
    history.append({'epoch':0,'dev':initial,'selected':True,'attempted_updates':0,'effective_updates':0})
    for epoch in range(1,s.epochs+1):
        logs=[]
        for window in windows(rows['train'],s.micro_batch,s.accumulation,s.seed+epoch):
            event=update_window(model,window,optimizer,scaler,device,dtype,pad_id)
            logs.append(event);attempted+=1;effective+=int(event['updated'])
        dev=evaluate_loss(model,rows['dev'],pad_id,device,dtype)
        chosen=selector.consider('dev',epoch,dev['loss'],model)
        train=evaluate_loss(model,rows['train'],pad_id,device,dtype)
        history.append({'epoch':epoch,'train':train,'dev':dev,'selected':chosen,
                        'attempted_updates':attempted,'effective_updates':effective,'windows':logs})
        write_json(directory/'history.json',history)
        print(f"epoch {epoch:02d} train={train['loss']:.5f} dev={dev['loss']:.5f} selected={chosen}")
    require(effective>0,'所有更新均跳过，不能标记训练成功')
    restore_adapter(model,selector.state)
    restored=evaluate_loss(model,rows['dev'],pad_id,device,dtype)
    require(math.isclose(restored['loss'],selector.loss,rel_tol=3e-4,abs_tol=3e-4),'最佳adapter恢复后dev指标不符')
    return {'selected_epoch':selector.epoch,'selected_dev_loss':selector.loss,
            'attempted_updates':attempted,'effective_updates':effective,'history':history}

In [6]:
# %% 05. 自由生成、严格评分与按ID配对。没有参考答案的请求才交给模型。

def score_answer(text,expected,stop_reason,error=None):
    try:
        actual=quantity_object(text);valid=error is None
    except (ValueError,TypeError,json.JSONDecodeError):
        actual=None;valid=False
    correct=valid and actual==quantity_object(expected)
    return {'format_ok':valid,'answer_correct':correct,'strict_success':correct and stop_reason=='eos' and error is None}


def aggregate(records,responses):
    expected_ids={r['id'] for r in records};require(len(expected_ids)==len(records),'计划题ID重复')
    by={}
    for response in responses:
        require(response['id'] in expected_ids and response['id'] not in by,'额外或重复响应ID')
        by[response['id']]=response
    scored=[]
    for record in records:
        result=by.get(record['id'],{'id':record['id'],'text':'','stop_reason':'missing','error':'missing'})
        score=score_answer(result.get('text',''),record['messages'][-1]['content'],result['stop_reason'],result.get('error'))
        scored.append(dict(result,**score,expected=record['messages'][-1]['content']))
    n=len(records);require(n>0,'计划题集不能为空')
    return {'planned':n,'received':len(responses),
            **{key+'_rate':sum(r[key] for r in scored)/n for key in ('format_ok','answer_correct','strict_success')},
            'rows':scored}


def paired(a,b):
    left={r['id']:r for r in a['rows']};right={r['id']:r for r in b['rows']}
    require(len(left)==len(a['rows']) and len(right)==len(b['rows']) and left.keys()==right.keys(),'比较题目不一致/重复')
    for k in left:
        require(left[k]['expected']==right[k]['expected'],'参考答案不一致')
        if 'request_sha256' in left[k] or 'request_sha256' in right[k]:
            require(left[k].get('request_sha256')==right[k].get('request_sha256'),'生成提示不一致')
    return {'improved':[k for k in left if not left[k]['strict_success'] and right[k]['strict_success']],
            'regressed':[k for k in left if left[k]['strict_success'] and not right[k]['strict_success']],
            'unchanged_success':[k for k in left if left[k]['strict_success'] and right[k]['strict_success']],
            'unchanged_failure':[k for k in left if not left[k]['strict_success'] and not right[k]['strict_success']]}


def generate_one(model,codec,record,s,device,dtype,real=False):
    ids=prompt_ids(codec,record)
    require(len(ids)<s.max_length,'提示太长；不截断历史')
    maximum=s.max_new_tokens
    if not real:
        maximum=min(maximum,s.max_length-len(ids))
    require(maximum>0,'没有生成空间')
    previous=model.training;model.eval()
    try:
        with torch.no_grad(),autocast(device,dtype):
            if real:
                from transformers import GenerationConfig
                eos=getattr(model.generation_config,'eos_token_id',None)
                eos=sorted(set(([eos] if type(eos) is int else eos or [])+[codec.eot_id]))
                cfg=GenerationConfig(do_sample=False,num_beams=1,max_new_tokens=maximum,
                                     pad_token_id=codec.pad_id,eos_token_id=eos,use_cache=True)
                inp=torch.tensor([ids],dtype=torch.long,device=device)
                seq=model.generate(input_ids=inp,attention_mask=torch.ones_like(inp),generation_config=cfg)
                require(seq[0,:len(ids)].tolist()==ids,'generate返回前缀不一致')
                new=seq[0,len(ids):].tolist()
            else:
                eos=[codec.eot_id];new=[]
                for _ in range(maximum):
                    inp=torch.tensor([ids+new],dtype=torch.long,device=device)
                    logits=model(input_ids=inp,attention_mask=torch.ones_like(inp),use_cache=False).logits
                    token=int(logits[0,-1].argmax());new.append(token)
                    if token in eos:break
        stop='eos' if new and new[-1] in eos else 'budget'
        body=new[:-1] if stop=='eos' else new
        if real:
            text=codec.tokenizer.decode(body,skip_special_tokens=False,clean_up_tokenization_spaces=False)
        else:
            text=codec.display(body)  # 不删除思考/角色/解释文本；严格评分器自行判错。
        return {'id':record['id'],'request_sha256':digest(ids),'input_ids':ids,'new_ids':new,
                'text':text,'stop_reason':stop,'error':None,'max_new_tokens':s.max_new_tokens}
    finally:model.train(previous)


def evaluate_generation(model,codec,records,s,device,dtype,path,real=False):
    responses=[]
    with open(path,'w',encoding='utf-8') as stream:
        for record in records:
            # 异常明确记录，缺失题保留分母。OOM不继续污染后续运行，写出后重新抛出。
            try:
                response=generate_one(model,codec,record,s,device,dtype,real)
            except Exception as exc:
                response={'id':record['id'],'text':'','stop_reason':'error','error':f'{type(exc).__name__}: {exc}',
                          'request_sha256':digest(prompt_ids(codec,record))}
                stream.write(json.dumps(response,ensure_ascii=False)+'\n');stream.flush();responses.append(response)
                if isinstance(exc,torch.cuda.OutOfMemoryError):raise
                continue
            responses.append(response);stream.write(json.dumps(response,ensure_ascii=False)+'\n');stream.flush()
    return aggregate(records,responses)


def probe_logits(model,rows,pad_id,device,dtype):
    previous=model.training;model.eval()
    batch=tensor_batch(rows[:1],pad_id,device)
    try:
        with torch.no_grad(),autocast(device,dtype):
            return model(input_ids=batch['input_ids'],attention_mask=batch['attention_mask'],use_cache=False).logits.float().cpu()
    finally:model.train(previous)

## 1. QLoRA 的计算对象

写出冻结的量化重建权重：

\[
\widehat W_0=\operatorname{dequant}(Q(W_0)),\qquad
Y=X\widehat W_0^\top+\frac{\alpha}{r}(XA^\top)B^\top.
\]

`Q(W0)` 的码与量化元数据被冻结；A/B 是可训练的浮点参数。它不是“对离散码做普通梯度下降”，也不是“所有运算都只有4位”。反传仍需穿过基座计算到达较早层的 adapter。[1][2]

与普通 LoRA 相比，改变的是基座的表示/计算路径，低秩分支的基本公式没有改变。本课不做 QAT，也不训练量化尺度。

In [7]:
# %% 06. NF4教学码本与真实打包；不模拟bitsandbytes性能。
# 数值码本：bitsandbytes 0.48.2 functional.get_4bit_type('nf4')。
NF4_VALUES = [-1.0, -0.6961928009986877, -0.5250730514526367,
              -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
              -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
              0.24611230194568634, 0.33791524171829224, 0.44070982933044434,
              0.5626170039176941, 0.7229568362236023, 1.0]

def pack_codes(codes):
    """本课格式：先出现的码放低4位，末尾填码7（代表零）；不兼容bnb布局。"""
    require(isinstance(codes,torch.Tensor) and codes.ndim==1 and codes.dtype==torch.uint8,'codes须一维uint8')
    require(codes.numel()>0 and bool((codes<=15).all()),'四位码须在0..15')
    if codes.numel()%2:
        codes=torch.cat([codes,torch.tensor([7],dtype=torch.uint8,device=codes.device)])
    return codes[0::2] | (codes[1::2]<<4)

def unpack_codes(packed,count):
    require(isinstance(packed,torch.Tensor) and packed.ndim==1 and packed.dtype==torch.uint8,'packed须一维uint8')
    require(type(count) is int and count>0 and packed.numel()==(count+1)//2,'码数/载荷长度不符')
    return torch.stack([packed&15,packed>>4],dim=-1).reshape(-1)[:count]

def quantize_nf4_teaching(weight,block_size=64):
    """扁平连续分块，FP32 absmax，最近码本项；不求量化过程的梯度。"""
    require(isinstance(weight,torch.Tensor) and weight.ndim==2 and weight.numel()>0,'权重须非空矩阵')
    require(weight.is_floating_point() and bool(torch.isfinite(weight).all()),'权重须有限浮点数')
    require(type(block_size) is int and block_size>0,'块大小须正整数')
    with torch.no_grad():
        flat=weight.detach().float().reshape(-1)
        padded=F.pad(flat,(0,(-flat.numel())%block_size)).reshape(-1,block_size)
        absmax=padded.abs().amax(dim=1)
        safe=torch.where(absmax==0,torch.ones_like(absmax),absmax)
        normalized=padded/safe[:,None]
        codebook=torch.tensor(NF4_VALUES,device=flat.device,dtype=torch.float32)
        codes=(normalized[...,None]-codebook).abs().argmin(dim=-1).to(torch.uint8).flatten()
        return {'packed':pack_codes(codes),'scales':absmax,'codebook':codebook,
                'shape':tuple(weight.shape),'block_size':block_size,'padded_count':codes.numel()}

def dequantize_nf4_teaching(package):
    code=unpack_codes(package['packed'],package['padded_count']).long()
    block=package['block_size'];shape=tuple(package['shape'])
    require(len(shape)==2 and all(type(x) is int and x>0 for x in shape),'形状无效')
    require(code.numel()%block==0 and package['scales'].numel()==code.numel()//block,'尺度数量不符')
    require(package['codebook'].shape==(16,) and bool(torch.isfinite(package['codebook']).all()),'码本无效')
    require(bool(torch.isfinite(package['scales']).all()) and bool((package['scales']>=0).all()),'尺度无效')
    result=package['codebook'][code].reshape(-1,block)*package['scales'].reshape(-1,1)
    require(math.prod(shape)<=result.numel(),'恢复形状越界')
    return result.flatten()[:math.prod(shape)].reshape(shape)

class PackedNF4Linear(nn.Module):
    """真正uint8打包的固定buffer，前向完整解码为FP32；明确不是融合四位内核。"""
    def __init__(self,linear,block_size=64):
        super().__init__()
        require(isinstance(linear,nn.Linear),'只接受干净nn.Linear')
        self.in_features=linear.in_features;self.out_features=linear.out_features
        package=quantize_nf4_teaching(linear.weight,block_size)
        self.block_size=block_size;self.padded_count=package['padded_count']
        for name in ('packed','scales','codebook'):
            self.register_buffer(name,package[name].clone())
        self.register_buffer('bias',None if linear.bias is None else linear.bias.detach().float().clone())
    def dequantized_weight(self):
        return dequantize_nf4_teaching({'packed':self.packed,'scales':self.scales,'codebook':self.codebook,
            'shape':(self.out_features,self.in_features),'block_size':self.block_size,'padded_count':self.padded_count})
    def forward(self,x):
        # 不能用no_grad包围F.linear：冻结权重仍要把梯度传给x。
        return F.linear(x,self.dequantized_weight().to(x.dtype),None if self.bias is None else self.bias.to(x.dtype))

def quantize_toy_blocks(model,block_size=64):
    require(isinstance(model,ToyLM),'教学量化只支持ToyLM')
    names=[n for n,m in model.named_modules() if isinstance(m,nn.Linear) and n.startswith('blocks.')]
    for name in names:
        parent,child=name.rsplit('.',1)
        setattr(model.get_submodule(parent),child,PackedNF4Linear(model.get_submodule(name),block_size))
    model.requires_grad_(False)
    return names

def frozen_state_hash(model):
    # state_dict也覆盖打包buffer、尺度、码本、真实bnb嵌套量化状态；只排除A/B。
    state=model.state_dict()
    pairs=[(name,tensor) for name,tensor in sorted(state.items())
           if 'lora_A' not in name and 'lora_B' not in name]
    return hash_parameters(pairs)

def payload_report(model):
    """模型张量元素载荷，不是进程内存/峰值显存。参数和buffer分别统计。"""
    params=list(model.parameters());buffers=list(model.buffers())
    return {'parameter_bytes':sum(p.numel()*p.element_size() for p in params),
            'buffer_bytes':sum(p.numel()*p.element_size() for p in buffers),
            'trainable_parameter_bytes':sum(p.numel()*p.element_size() for p in params if p.requires_grad),
            'packed_nf4_modules':sum(isinstance(m,PackedNF4Linear) for m in model.modules()),
            'scope':'tensor payload only; offline forward materializes FP32 weights; not peak memory'}

def adapter_only_update(model,x,target,optimizer):
    assert_optimizer(model,optimizer);optimizer.zero_grad(set_to_none=True)
    loss=F.mse_loss(model(x),target);loss.backward()
    norms={n:None if p.grad is None else float(p.grad.norm()) for n,p in adapter_parameters(model)}
    require(all(v is not None and math.isfinite(v) for v in norms.values()),'adapter梯度缺失/非有限')
    optimizer.step()
    return float(loss.detach()),norms

## 2. NF4 是码本，不是均匀整数刻度

本课采用 bitsandbytes 公开的 16 个 NF4 代表值，先按块 `absmax` 归一化，再寻找最近的码本项。[3]

**离线格式是教学实现**：展平后连续分块、FP32 尺度、两码放一字节；先出现的码放低4位。它没有实现双重量化、融合算子或分页优化器，也不能作为 bitsandbytes 权重文件使用。前向会生成整层 FP32 重建矩阵，因此不提供 GPU 性能保证。

看码本相邻间隔，确认它不是 `[-7,...,7]` 的均匀刻度。

In [8]:
codebook = torch.tensor(NF4_VALUES)
print("NF4码本：", codebook.tolist())
print("相邻间隔：", (codebook[1:] - codebook[:-1]).tolist())
w = torch.tensor([[-1., -.3, -.1, 0., .1, .3, 1.]])
package = quantize_nf4_teaching(w, block_size=8)
print("原权重：", w.tolist())
print("打包字节：", package["packed"].tolist())
print("恢复值：", dequantize_nf4_teaching(package).tolist())
print("注意：最后一个零是分块填充，不是原矩阵的新参数。")

NF4码本： [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]
相邻间隔： [0.30380719900131226, 0.17111974954605103, 0.1301555633544922, 0.11047610640525818, 0.09966795146465302, 0.09372339397668839, 0.09105003625154495, 0.07958029955625534, 0.08134990185499191, 0.08518210053443909, 0.0918029397726059, 0.1027945876121521, 0.12190717458724976, 0.1603398323059082, 0.2770431637763977]
原权重： [[-1.0, -0.30000001192092896, -0.10000000149011612, 0.0, 0.10000000149011612, 0.30000001192092896, 1.0]]
打包字节： [64, 118, 184, 127]
恢复值： [[-1.0, -0.28444138169288635, -0.09105003625154495, 0.0, 0.07958029955625534, 0.33791524171829224, 1.0]]
注意：最后一个零是分块填充，不是原矩阵的新参数。


## 3. 冻结基座 ≠ 关闭计算图

`PackedNF4Linear` 没有可训练参数，只有码、尺度等 buffer。但 `F.linear(x, weight_hat)` 仍需要为输入 `x` 传播梯度。不要把它整个放进 `no_grad()`；否则上游可训练 adapter 的梯度会被切断。

B 为零时，初始增量为零；应当与**同一量化基座**比较，而不是要求初始输出等于原浮点基座。先预测下面两种最大差异，哪个应该为零？

In [9]:
def rank_probe(steps=350):
    """线性机制实验：原目标增量rank=2，但量化误差一般不是rank=2。不是SFT成绩。"""
    torch.manual_seed(360)
    linear=nn.Linear(16,12,bias=False);linear.requires_grad_(False)
    u,_=torch.linalg.qr(torch.randn(12,2));v,_=torch.linalg.qr(torch.randn(16,2))
    delta=(u*torch.tensor([0.4,0.2]))@v.T
    x=torch.eye(16)*4
    target=F.linear(x,linear.weight.detach()+delta)
    test_x=torch.randn(128,16);test_y=F.linear(test_x,linear.weight.detach()+delta)
    packed=PackedNF4Linear(linear)
    result={}
    for name,base in [('floating_base',copy.deepcopy(linear)),('teaching_nf4_base',packed)]:
        torch.manual_seed(361)
        model=ToyLoRALinear(base,2,2);base.requires_grad_(False)
        before=frozen_state_hash(model)
        opt=torch.optim.Adam([p for _,p in adapter_parameters(model)],lr=0.02)
        with torch.no_grad():initial=float(F.mse_loss(model(x),target))
        first=None
        for i in range(steps):
            loss,norms=adapter_only_update(model,x,target,opt)
            if i==0:first=norms
        with torch.no_grad():
            final=float(F.mse_loss(model(x),target));test=float(F.mse_loss(model(test_x),test_y))
        result[name]={'initial_mse':initial,'final_mse':final,'heldout_input_mse':test,
            'base_unchanged':before==frozen_state_hash(model),'first_grad_norms':first,
            'trainable_parameters':sum(p.numel() for _,p in adapter_parameters(model))}
    quant_error=linear.weight.detach()-packed.dequantized_weight()
    result['quantization_error_rank']=int(torch.linalg.matrix_rank(quant_error))
    result['original_task_update_rank']=int(torch.linalg.matrix_rank(delta))
    result['limitations']='fixed synthetic low-rank linear target; not pretrained LM; no bnb; no quality ranking'
    return result

def mechanism_report():
    torch.manual_seed(362)
    layer=nn.Linear(16,12,bias=False);layer.requires_grad_(False)
    packed=PackedNF4Linear(layer,64);adapter=ToyLoRALinear(packed,2,4)
    x=torch.randn(4,16,requires_grad=True);target=torch.randn(4,12)
    base_output=packed(x);original=layer(x);initial=adapter(x)
    initial_diff=float((base_output-initial).detach().abs().max())
    quant_diff=float((base_output-original).detach().abs().max())
    before=frozen_state_hash(adapter)
    opt=torch.optim.Adam([p for _,p in adapter_parameters(adapter)],lr=0.02)
    _,first=adapter_only_update(adapter,x,target,opt)
    input_grad=float(x.grad.norm());x.grad=None
    _,second=adapter_only_update(adapter,x,target,opt)
    # 错误顺序的演示：模拟prepare的freeze-all副作用，不调用/冒充PEFT。
    wrong=copy.deepcopy(adapter);wrong.requires_grad_(False)
    frozen_adapters=sum(p.numel() for p in wrong.parameters() if p.requires_grad)
    # 非交换性：训练后合并到原浮点基座不等于合并到反量化基座。
    delta=adapter.scale*(adapter.lora_B@adapter.lora_A)
    effective=packed.dequantized_weight()+delta
    wrong_merge=layer.weight.detach()+delta
    re_quant=dequantize_nf4_teaching(quantize_nf4_teaching(effective))
    return {'kind':'pure_torch_nf4_codebook_demo_NOT_bitsandbytes',
        'initial_vs_quantized_base_max_diff':initial_diff,'initial_vs_original_base_max_diff':quant_diff,
        'first_grad_norms':first,'second_grad_norms':second,'input_grad_norm':input_grad,
        'frozen_codes_scales_and_bias_unchanged':before==frozen_state_hash(adapter),
        'trainable_after_wrong_freeze_all':frozen_adapters,
        'wrong_base_merge_weight_max_diff':float((effective-wrong_merge).detach().abs().max()),
        'requantize_merged_weight_max_diff':float((re_quant-effective).detach().abs().max()),
        'double_quant_memory_math':{'weight_bits':4,'fp32_scale_bits_per_weight_block64':32/64,
             'illustrative_nested_scale_bits':8/64+32/(64*256),
             'assumptions':'ignore padding, offset/codebooks/headers; illustrative 8-bit scale + FP32 secondary scale; not exact bnb footprint'},
        'packed_payload':payload_report(adapter)}

In [10]:
mechanisms = mechanism_report()
for key in ("initial_vs_quantized_base_max_diff", "initial_vs_original_base_max_diff",
            "first_grad_norms", "second_grad_norms", "input_grad_norm",
            "frozen_codes_scales_and_bias_unchanged"):
    print(key, ":", mechanisms[key])

initial_vs_quantized_base_max_diff : 0.0
initial_vs_original_base_max_diff : 0.1148049384355545
first_grad_norms : {'lora_A': 0.0, 'lora_B': 0.6183766722679138}
second_grad_norms : {'lora_A': 0.11570838838815689, 'lora_B': 0.600257158279419}
input_grad_norm : 0.2026059925556183
frozen_codes_scales_and_bias_unchanged : True


首步 A 梯度为 0、B 梯度非零是正常初始化现象；B 更新后 A 通常可以获得梯度。本例输入梯度非零，量化基座状态保持不变。

**本例没有用 STE 近似“穿过舍入求导”**。量化在训练前完成；反传使用固定重建权重，求输入与 A/B 的梯度。不要把“通过量化层反传”和“训练整数码”混为一谈。

## 4. 准备与注入顺序：反过来会发生什么？

课程固定的 PEFT 0.18.0 `prepare_model_for_kbit_training` 会冻结现有参数，并将非 `Params4bit` 的 FP16/BF16 参数升为 FP32；可选启用非重入激活检查点。[4]

正确顺序：**加载量化基座 → prepare → 注入 LoRA → 检查可训练集合 → 创建优化器。**

下面只模拟 freeze-all 副作用，不冒充真实 PEFT 执行。

In [11]:
torch.manual_seed(370)
wrong_order = ToyLoRALinear(PackedNF4Linear(nn.Linear(7, 5)), rank=2, alpha=4)
print("注入后的可训练数量：", sum(p.numel() for p in wrong_order.parameters() if p.requires_grad))
wrong_order.requires_grad_(False)  # 模拟在注入后执行freeze-all
print("错误冻结后数量：", sum(p.numel() for p in wrong_order.parameters() if p.requires_grad))
try:
    adapter_parameters(wrong_order)
except ValueError as error:
    print("审计正确拒绝：", error)

注入后的可训练数量： 24
错误冻结后数量： 0
审计正确拒绝： 存在非adapter可训练参数或全部冻结


## 5. 双重量化与分页优化器分别做什么？

QLoRA 原论文包含 NF4、对量化常数再次量化，以及用统一内存分页缓解内存峰值的优化器。[1] **双重量化不等于给权重再乘一次四位比例**。

假设每64个权重使用一个FP32尺度，尺度摊销为 `32/64=0.5 bit/weight`。若第一层尺度用8位，并以256个尺度一组保存一个FP32二级尺度，理想化开销变为 `8/64 + 32/(64*256)`。这只是示意，未计 offset、码本、填充与文件元数据，不是实际 bnb 文件大小。[3]

本课真实路径启用 NF4 + 双重量化，但**使用普通 `torch.optim.AdamW`，不是分页优化器**。这里的 QLoRA 教学路径指冻结四位基座上的 LoRA 训练，不宣称复现论文所有设置。

In [12]:
plain_scale_bits = 32 / 64
nested_scale_bits = 8 / 64 + 32 / (64 * 256)
print("单层尺度开销 bit/weight：", plain_scale_bits)
print("理想化嵌套尺度开销：", nested_scale_bits)
print("理想化总位数：", 4 + nested_scale_bits)
print("减少：", plain_scale_bits - nested_scale_bits, "bit/weight（不是实际显存测量）")

单层尺度开销 bit/weight： 0.5
理想化嵌套尺度开销： 0.126953125
理想化总位数： 4.126953125
减少： 0.373046875 bit/weight（不是实际显存测量）


## 6. 量化误差会被 LoRA 自动完全修复吗？

一般没有这种保证。设原任务希望增加 $\Delta W_*$，量化后要达到同一个有效矩阵，需要拟合：

\[
\Delta W_* + (W_0-\widehat W_0).
\]

即使原任务增量秩很低，加上量化误差以后也未必低秩。下面用已知 rank=2 的线性任务验证：两个分支使用相同 A/B 初始化、rank、样本、优化器和步数，分别在原浮点基座与教学量化基座上训练。

这是机制反例，不是语言模型效果排名。

In [13]:
rank_results = rank_probe()
print(json.dumps(rank_results, ensure_ascii=False, indent=2))

{
  "floating_base": {
    "initial_mse": 0.016666671261191368,
    "final_mse": 5.420135451572605e-09,
    "heldout_input_mse": 5.7284910148780455e-09,
    "base_unchanged": true,
    "first_grad_norms": {
      "lora_A": 0.0,
      "lora_B": 0.011705908924341202
    },
    "trainable_parameters": 56
  },
  "teaching_nf4_base": {
    "initial_mse": 0.018415866419672966,
    "final_mse": 0.0018050679937005043,
    "heldout_input_mse": 0.0017898321384564042,
    "base_unchanged": true,
    "first_grad_norms": {
      "lora_A": 0.0,
      "lora_B": 0.012059634551405907
    },
    "trainable_parameters": 56
  },
  "quantization_error_rank": 12,
  "original_task_update_rank": 2,
  "limitations": "fixed synthetic low-rank linear target; not pretrained LM; no bnb; no quality ranking"
}


观察：原任务更新秩为2，但量化误差可能为更高秩。当前固定 rank=2 的 adapter 可以降低任务损失，却未必消除全部额外误差。不能据此把“增加 rank”当作任何真实任务的唯一解决办法。

下面开始连接真实模型。源码内置，执行定义本身不会联网。

In [14]:
# %% 07. 真实NF4加载、准备与PEFT注入。明确CUDA，无CPU回退。
def require_real():
    wanted={'transformers':'4.57.3','peft':'0.18.0','accelerate':'1.12.0','bitsandbytes':'0.48.2'}
    for name,version in wanted.items():
        try:actual=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError as exc:
            raise RuntimeError(f'真实路径缺少{name}=={version}；请在独立环境安装；不会回退教学模型') from exc
        require(actual==version,f'课程固定接口需要{name}=={version}，实际{actual}；不自动升级')
    import transformers,peft,bitsandbytes
    return transformers,peft,bitsandbytes

def quantization_spec(s):
    require(s.dtype in ('float16','bfloat16'),'真实NF4计算精度须FP16/BF16')
    return {'load_in_4bit':True,'bnb_4bit_quant_type':'nf4','bnb_4bit_use_double_quant':True,
            'bnb_4bit_compute_dtype':s.dtype,'bnb_4bit_quant_storage':'uint8',
            'llm_int8_skip_modules':['lm_head'],'llm_int8_enable_fp32_cpu_offload':False}

def nf4_config(s):
    tr,_,_=require_real();fields=quantization_spec(s)
    fields['bnb_4bit_compute_dtype']=getattr(torch,fields['bnb_4bit_compute_dtype'])
    fields['bnb_4bit_quant_storage']=torch.uint8
    return tr.BitsAndBytesConfig(**fields)

def cuda_memory(device):
    if device.type!='cuda':
        return {'allocated':None,'reserved':None,'peak_allocated':None,'device':'cpu'}
    torch.cuda.synchronize(device)
    return {'allocated':torch.cuda.memory_allocated(device),'reserved':torch.cuda.memory_reserved(device),
            'peak_allocated':torch.cuda.max_memory_allocated(device),'device':str(device)}

def load_nf4_base(s,training=True):
    tr,peft,_=require_real();device,dtype=runtime(s)
    require(device.type=='cuda','本课真实QLoRA只实现单CUDA设备；不静默CPU回退')
    require(s.max_length+s.max_new_tokens<=40960,'课堂预算过大；需要单独分析长上下文资源')
    stages={}
    base=tr.AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,
        dtype=dtype,quantization_config=nf4_config(s),device_map={'':0},
        low_cpu_mem_usage=True,attn_implementation='sdpa',trust_remote_code=False,
        local_files_only=s.local_files_only)
    stages['after_load']=cuda_memory(device)
    require(getattr(base,'is_loaded_in_4bit',False),'未实际四位加载')
    require(all(p.device.type=='cuda' and p.device.index==0 for p in base.parameters()),'发现CPU/其他设备卸载')
    base.config.use_cache=False
    # 必须在注入adapter之前。该版本还会将非Params4bit的半精度参数升为FP32。
    base=peft.prepare_model_for_kbit_training(base,
        use_gradient_checkpointing=bool(training and s.gradient_checkpointing),
        gradient_checkpointing_kwargs={'use_reentrant':False})
    require(not any(p.requires_grad for p in base.parameters()),'prepare后仍有基座可训练参数')
    stages['after_prepare']=cuda_memory(device)
    return base,stages

def target_plan(base,s):
    cfg=base.config;d=cfg.hidden_size;hd=getattr(cfg,'head_dim',d//cfg.num_attention_heads)
    names=('q_proj','v_proj') if s.targets=='qv' else ('q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj')
    expected={}
    for i in range(cfg.num_hidden_layers):
        dimensions={'q_proj':(d,cfg.num_attention_heads*hd),'k_proj':(d,cfg.num_key_value_heads*hd),
            'v_proj':(d,cfg.num_key_value_heads*hd),'o_proj':(cfg.num_attention_heads*hd,d),
            'gate_proj':(d,cfg.intermediate_size),'up_proj':(d,cfg.intermediate_size),'down_proj':(cfg.intermediate_size,d)}
        for name in names:
            prefix='self_attn' if name in ('q_proj','k_proj','v_proj','o_proj') else 'mlp'
            expected[f'model.layers.{i}.{prefix}.{name}']=dimensions[name]
    return expected

def inject_nf4_lora(base,s):
    _,peft,bnb=require_real();require(not hasattr(base,'peft_config'),'禁止重复注入')
    plan=target_plan(base,s)
    for name,(inputs,outputs) in plan.items():
        module=base.get_submodule(name)
        require(isinstance(module,bnb.nn.Linear4bit),'目标不是实际bnb Linear4bit：'+name)
        require((module.in_features,module.out_features)==(inputs,outputs),'投影维度不符：'+name)
    targets=['q_proj','v_proj'] if s.targets=='qv' else ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
    config=peft.LoraConfig(task_type=peft.TaskType.CAUSAL_LM,target_modules=targets,r=s.rank,
        lora_alpha=s.alpha,lora_dropout=0.0,bias='none',init_lora_weights=True,use_rslora=False,inference_mode=False)
    model=peft.get_peft_model(base,config,autocast_adapter_dtype=True,revision=REVISION)
    model.peft_config['default'].base_model_name_or_path=MODEL_ID;model.peft_config['default'].revision=REVISION
    wanted=set()
    for name,(inputs,outputs) in plan.items():
        m=model.get_base_model().get_submodule(name)
        a,b=m.lora_A['default'].weight,m.lora_B['default'].weight
        require(a.shape==(s.rank,inputs) and b.shape==(outputs,s.rank),'A/B形状错误')
        require(a.dtype==b.dtype==torch.float32,'本课adapter必须FP32')
        require(not torch.count_nonzero(b),'初始化B不是零')
        wanted.update((id(a),id(b)))
    actual={id(p) for _,p in adapter_parameters(model)}
    require(actual==wanted,'可训练集合不等于目标A/B')
    return model,{'paths':list(plan),'expected_trainable':sum(s.rank*(i+o) for i,o in plan.values())}

def audit_quantized_layers(model,s):
    _,_,bnb=require_real();rows=[]
    for name,layer in model.named_modules():
        if isinstance(layer,bnb.nn.Linear4bit):
            state=layer.weight.quant_state
            require(state is not None,'缺少量化状态')
            require(state.quant_type=='nf4' and bool(state.nested),'不是NF4双重量化')
            require(layer.weight.dtype==torch.uint8 and not layer.weight.requires_grad,'基座存储或冻结不符')
            require(layer.compute_dtype==getattr(torch,s.dtype),'量化层计算精度不符')
            rows.append({'path':name,'storage_dtype':str(layer.weight.dtype),'compute_dtype':str(layer.compute_dtype),
                         'nested':bool(state.nested),'block_size':state.blocksize,
                         'packed_shape':list(layer.weight.shape),'logical_shape':list(state.shape)})
    require(bool(rows),'未发现真实Linear4bit')
    base=model.get_base_model() if hasattr(model,'peft_config') else model
    require(len(rows)==base.config.num_hidden_layers*7,'预期每层7个量化投影，实际不符')
    return {'quantized_modules':rows,'count':len(rows),
        'parameter_dtypes':sorted({str(p.dtype) for p in model.parameters()}),
        'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
        'library_footprint_bytes':int(model.get_memory_footprint())}

## 7. 真实配置：三个 dtype、两类参数

| 对象 | 本课真实路径 |
|---|---|
| 基座线性权重 | NF4四位码，`uint8`容器 |
| 量化统计 | 双重量化；必须存在嵌套量化状态 |
| Linear4bit计算 | FP16，或显式选择BF16 |
| LoRA参数 | FP32 |
| 部分未量化权重 | prepare后可能FP32，例如Embedding/归一化/输出头 |
| 优化器 | 只持有LoRA参数，普通AdamW |

四位存储不意味着激活、梯度、优化器状态、KV cache 都是四位。也不能只看模型的第一个参数判断量化是否成功。[2][4][5]

`device_map={"":0}` 是固定单GPU，不使用推理导向的自动卸载。本课不调用整个量化模型的 `.half()` 或 `.to(...)` 来“改精度”。

In [15]:
real_config = Settings(device="cuda", dtype="float16", epochs=3, micro_batch=1,
                       lr=2e-4, gradient_checkpointing=True)
print(json.dumps(quantization_spec(real_config), ensure_ascii=False, indent=2))
print("仅生成配置字典，尚未加载模型，也没有CUDA调用。")

{
  "load_in_4bit": true,
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "llm_int8_skip_modules": [
    "lm_head"
  ],
  "llm_int8_enable_fp32_cpu_offload": false
}
仅生成配置字典，尚未加载模型，也没有CUDA调用。


### 核心顺序（完整代码在上面的真实实现中）

```python
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION,
    quantization_config=quant_config,
    dtype=torch.float16, device_map={"": 0},
)
base.config.use_cache = False
base = prepare_model_for_kbit_training(
    base, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(base, lora_config, autocast_adapter_dtype=True)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4
)
```

默认只训练Q/V，便于控制与前课的目标范围；并不是原论文all-linear设置的完整复刻。`--targets all-linear` 会明确覆盖每层七类投影，但不包含词表输出头。改变目标范围是新实验，不能把效果差异只归于量化。[6]

In [16]:
# 固定Qwen配置的预算推导，不是加载后测量；真实运行还会逐层核对。
fixed_config = SimpleNamespace(hidden_size=1024, num_attention_heads=16,
    num_key_value_heads=8, head_dim=128, intermediate_size=3072, num_hidden_layers=28)
for choice in ["qv", "all-linear"]:
    plan = target_plan(SimpleNamespace(config=fixed_config), Settings(targets=choice))
    print(choice, "目标模块：", len(plan), "r=8参数：", sum(8*(i+o) for i,o in plan.values()))

qv 目标模块： 56 r=8参数： 1146880
all-linear 目标模块： 196 r=8参数： 5046272


## 8. 数据与监督：继续沿用第35课，不更换评分规则

24个来源、每来源两种问法，共48条对话，固定划分16/4/4个来源，即32/8/8条训练/开发/测试。只监督最后一条assistant回复和结束符。

训练前没有把测试答案放入生成请求。选择模型只读开发集；epoch 0也有资格胜出。评测时保留所有8题，无输出、格式错误、预算耗尽都不从分母删除。

真实Qwen调用不传`labels`，在外部手动对齐一次logits/labels，再以窗口总监督token数加权。不要既手动移位又调用内部移位的loss。

In [17]:
parts = split_three(make_records())
print(validate_splits(parts))
codec = ToyChatCodec()
row = codec.encode(parts["train"][0], policy="last")
print("输入token数：", len(row["input_ids"]))
print("监督token数：", sum(y != IGNORE for y in row["labels"][1:]))
print("末轮参考答案（只在训练/评分侧）：", parts["train"][0]["messages"][-1]["content"])
print("生成请求最后一条角色：", public_messages(parts["train"][0])[-1]["role"])

{'train': {'records': 32, 'groups': 16}, 'dev': {'records': 8, 'groups': 4}, 'test': {'records': 8, 'groups': 4}}
输入token数： 162
监督token数： 15
末轮参考答案（只在训练/评分侧）： {"quantity":2}
生成请求最后一条角色： user


## 9. 完整运行器：训练、保存与同量化契约重载

以下实现将上面的组件连接起来。不同于普通LoRA，它还保存量化方案、双重量化设置、计算dtype、基座revision、依赖版本、模板与数据指纹。

冻结检查覆盖**量化码、尺度、码本和其余固定状态**，不是只看可训练参数的数量。基准是同一量化基座关闭adapter，不是原FP16基座。

真实路径记录加载、prepare、注入、fit前后的内存。fit峰值包括训练及训练/开发损失核对，不包括最初加载和最终生成；它不是服务进程显存，也不是速度基准。[7]

In [18]:
# %% 08. 训练执行与重载契约：相同量化基座，不偷偷合并到原浮点权重。
def inspect_pipeline(s,records=None):
    directory=new_run(s.output_dir,'inspect')
    manifest={'status':'started','kind':'tokenizer_only','environment':environment()}
    write_json(directory/'manifest.json',manifest)
    try:
        records=make_records() if records is None else records
        parts=split_three(records,s.seed);codec=QwenChatCodec(s.local_files_only)
        rows=prepare_splits(parts,codec,s.max_length)
        info={'codec':codec.contract(),'split':validate_splits(parts),'quantization':quantization_spec(s),
              'sample':rows['train'][0],'lengths':{k:[len(r['input_ids']) for r in v] for k,v in rows.items()},
              'targets':s.targets,'expected_qv_rank8_parameters':1146880,
              'caution':'只检查分词器；没有量化加载、训练或GPU测试'}
        write_json(directory/'inspection.json',info)
        manifest['status']='completed';write_json(directory/'manifest.json',manifest)
        print('分词器检查：',directory);return directory
    except Exception as exc:
        manifest.update(status='failed',error=f'{type(exc).__name__}: {exc}');write_json(directory/'manifest.json',manifest);raise

def make_toy_model(s,quantized=True,raw_state=None):
    require(s.targets=='qv','教学网络只实现qv；all-linear仅用于真实Qwen')
    torch.manual_seed(s.seed);base=ToyLM(s.max_length)
    if raw_state is not None:base.load_state_dict(raw_state,strict=True)
    if quantized:quantize_toy_blocks(base,s.block_size)
    base.requires_grad_(False);base.checkpointed=s.gradient_checkpointing
    torch.manual_seed(s.seed+1000);model=ToyAdapter(base,s.rank,s.alpha)
    return model

def write_adapter(model,directory,real,s,codec,parts,frozen_hash,raw_state=None):
    folder=directory/'adapter';folder.mkdir(exist_ok=False)
    if real:
        model.save_pretrained(str(folder),safe_serialization=True,save_embedding_layers=False)
        from safetensors.torch import load_file
        saved=load_file(str(folder/'adapter_model.safetensors'))
        require(bool(saved) and all('lora_A' in n or 'lora_B' in n for n in saved),'保存了非A/B内容或空adapter')
        codec.tokenizer.save_pretrained(str(directory/'tokenizer'))
    else:
        torch.save({'format':FORMAT,'state':snapshot_adapter(model)},folder/'teaching_adapter.pt')
        torch.save({'format':FORMAT,'raw_state':raw_state},directory/'teaching_base.pt')
    contract={'format':FORMAT,'kind':'real_bnb_nf4_peft' if real else 'teaching_nf4_FP32_dequant',
        'settings':asdict(s),'base_id':MODEL_ID if real else 'random_ToyLM',
        'revision':REVISION if real else None,'codec':codec.contract(),
        'quantization':quantization_spec(s) if real else {'type':'nf4_codebook_teaching','double_quant':False,
            'block_size':s.block_size,'storage':'low_nibble_first_custom','compute':'full_FP32_dequant'},
        'environment':environment(),'frozen_state_sha256':frozen_hash,
        'data_sha256':digest(parts),'selection':'dev_loss only; test not used',
        'resume_complete':False,'optimizer':'ordinary torch AdamW, NOT paged optimizer'}
    write_json(directory/'contract.json',contract)
    return contract

def reload_run(directory):
    directory=Path(directory).resolve();contract=json.loads((directory/'contract.json').read_text(encoding='utf-8'))
    require(contract['format']==FORMAT,'不是本课检查点')
    s=Settings(**contract['settings']);device,dtype=runtime(s)
    real=contract['kind']=='real_bnb_nf4_peft'
    if real:
        require(contract['base_id']==MODEL_ID and contract['revision']==REVISION,'基座来源被改变')
        require(quantization_spec(s)==contract['quantization'],'量化配置被改变')
        current=environment()
        require(current['versions']==contract['environment']['versions'],'重载软件版本变化，先建新实验而非声称精确恢复')
        require(current['cuda_device']==contract['environment']['cuda_device'],'重载GPU变化，需重新评测')
        _,peft,_=require_real();codec=QwenChatCodec(s.local_files_only)
        base,_=load_nf4_base(s,training=False)
        model=peft.PeftModel.from_pretrained(base,str(directory/'adapter'),is_trainable=False,autocast_adapter_dtype=True)
    else:
        codec=ToyChatCodec()
        payload=torch.load(directory/'teaching_base.pt',map_location='cpu',weights_only=True)
        require(payload['format']==FORMAT,'基座格式错误')
        model=make_toy_model(s,True,payload['raw_state']).to(device)
        payload=torch.load(directory/'adapter'/'teaching_adapter.pt',map_location='cpu',weights_only=True)
        require(payload['format']==FORMAT,'adapter格式错误');restore_adapter(model,payload['state'])
        model.requires_grad_(False)
    require(codec.contract()==contract['codec'],'tokenizer或模板变化')
    require(frozen_state_hash(model)==contract['frozen_state_sha256'],'基座/打包码/尺度发生变化')
    probe=json.loads((directory/'probe.json').read_text(encoding='utf-8'))
    expected=torch.load(directory/'probe_logits.pt',map_location='cpu',weights_only=True)
    got=probe_logits(model,[probe['row']],codec.pad_id,device,dtype)
    atol,rtol=(2e-2,1e-2) if real else (1e-6,1e-5)
    torch.testing.assert_close(got,expected,atol=atol,rtol=rtol)
    generation=generate_one(model,codec,probe['record'],s,device,dtype,real)
    same=generation['new_ids']==probe['generation']['new_ids']
    result={'frozen_state_verified':True,'logits_max_diff':float((got-expected).abs().max()),
            'same_greedy_ids':same,'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
            'probe_split':'dev','atol':atol,'rtol':rtol,'generation':generation}
    write_json(directory/'reload.json',result)
    require(same,'探针生成分叉；不伪称精确恢复，检查接近并列与环境')
    require(result['trainable_parameters']==0,'推理重载仍有可训练参数')
    print('相同量化基座重载：',result['logits_max_diff'],'；生成相同：',same)
    return result

def run_pipeline(s=None,real=False,records=None):
    s=s or Settings();device,dtype=runtime(s)
    if real:require(device.type=='cuda','真实路径要求CUDA；不回退')
    else:require(device.type=='cpu','教学实验固定CPU，避免把它误当CUDA低比特基准')
    directory=new_run(s.output_dir,'real' if real else 'offline')
    manifest={'status':'started','stage':'data','environment':environment(),'settings':asdict(s),
              'kind':'actual_Qwen_bnb_PEFT' if real else 'random_toy_NF4_codebook_FP32_dequant',
              'real_weights_executed':False,'cuda_executed':False}
    write_json(directory/'manifest.json',manifest)
    try:
        torch.set_num_threads(1);random.seed(s.seed);torch.manual_seed(s.seed)
        records=make_records() if records is None else records
        parts=split_three(records,s.seed);codec=QwenChatCodec(s.local_files_only) if real else ToyChatCodec()
        rows=prepare_splits(parts,codec,s.max_length)
        for name in parts:
            write_jsonl(directory/f'{name}_messages.jsonl',parts[name]);write_jsonl(directory/f'{name}_tokens.jsonl',rows[name])
        write_json(directory/'split.json',{'summary':validate_splits(parts),'sha256':digest(parts)})
        raw_state=None;memory={}
        manifest['stage']='load_prepare_inject';write_json(directory/'manifest.json',manifest)
        if real:
            base,memory=load_nf4_base(s)
            baseline_probe=probe_logits(base,rows['dev'],codec.pad_id,device,dtype)
            model,plan=inject_nf4_lora(base,s)
            initial=probe_logits(model,rows['dev'],codec.pad_id,device,dtype)
            quant_audit=audit_quantized_layers(model,s)
            memory['after_lora']=cuda_memory(device)
        else:
            torch.manual_seed(s.seed);raw=ToyLM(s.max_length)
            raw_state={n:t.detach().cpu().clone() for n,t in raw.state_dict().items()}
            raw_probe=probe_logits(raw,rows['dev'],codec.pad_id,device,dtype)
            model=make_toy_model(s,True,raw_state)
            with model.disable_adapter():baseline_probe=probe_logits(model,rows['dev'],codec.pad_id,device,dtype)
            initial=probe_logits(model,rows['dev'],codec.pad_id,device,dtype)
            plan={'targets':'qv','paths':[]};quant_audit=payload_report(model)
            quant_audit['original_float_vs_quantized_logits_max_diff']=float((raw_probe-baseline_probe).abs().max())
            del raw,raw_probe
        torch.testing.assert_close(initial,baseline_probe,atol=2e-3 if real else 1e-7,rtol=1e-4)
        frozen_before=frozen_state_hash(model)
        audit={'plan':plan,'quantized_layers':quant_audit,'frozen_state_before':frozen_before,
               'initial_vs_quantized_base_logits_max_diff':float((initial-baseline_probe).abs().max()),
               'trainable_parameters':sum(p.numel() for _,p in adapter_parameters(model))}
        write_json(directory/'audit.json',audit)
        manifest['stage']='train_dev_select';write_json(directory/'manifest.json',manifest)
        memory['before_fit']=cuda_memory(device)
        if real:torch.cuda.reset_peak_memory_stats(device)
        training=fit(model,{k:rows[k] for k in ('train','dev')},s,codec.pad_id,device,dtype,directory)
        memory['after_fit']=cuda_memory(device)
        memory['training_peak_increment']=None if not real else memory['after_fit']['peak_allocated']-memory['before_fit']['allocated']
        memory['scope']='fit only: forward, backward, clipping, optimizer, train/dev checks; excludes loading and final generation; PyTorch allocated, not nvidia-smi'
        require(frozen_state_hash(model)==frozen_before,'冻结基座/码本/尺度被改写')
        with model.disable_adapter():disabled=probe_logits(model,rows['dev'],codec.pad_id,device,dtype)
        torch.testing.assert_close(disabled,baseline_probe,atol=2e-3 if real else 1e-7,rtol=1e-4)
        audit.update(frozen_state_unchanged=True,disabled_vs_quantized_base_logits_max_diff=float((disabled-baseline_probe).abs().max()))
        write_json(directory/'audit.json',audit);write_json(directory/'memory.json',memory)
        write_json(directory/'selection.json',{k:v for k,v in training.items() if k!='history'})
        contract=write_adapter(model,directory,real,s,codec,parts,frozen_before,raw_state)
        manifest['stage']='locked_test';write_json(directory/'manifest.json',manifest)
        with model.disable_adapter():
            before_loss=evaluate_loss(model,rows['test'],codec.pad_id,device,dtype)
            before_eval=evaluate_generation(model,codec,parts['test'],s,device,dtype,directory/'base_raw.jsonl',real)
        after_loss=evaluate_loss(model,rows['test'],codec.pad_id,device,dtype)
        after_eval=evaluate_generation(model,codec,parts['test'],s,device,dtype,directory/'adapter_raw.jsonl',real)
        report={'kind':manifest['kind'],'baseline':'same quantized base, adapter disabled; NOT original FP16 base',
                'base':{'teacher_forced':before_loss,'generation':before_eval},
                'adapter':{'teacher_forced':after_loss,'generation':after_eval},'paired':paired(before_eval,after_eval),
                'selected_epoch':training['selected_epoch'],'limits':'single seed, template data, 8 heldout examples; no language capability or speed claims'}
        write_json(directory/'report.json',report)
        expected=probe_logits(model,rows['dev'],codec.pad_id,device,dtype)
        gen=generate_one(model,codec,parts['dev'][0],s,device,dtype,real)
        write_json(directory/'probe.json',{'record':parts['dev'][0],'row':rows['dev'][0],'generation':gen})
        torch.save(expected,directory/'probe_logits.pt')
        del model,initial,baseline_probe,disabled,expected
        if real:del base
        gc.collect()
        if real:torch.cuda.empty_cache()
        manifest['stage']='fresh_quantized_base_reload';write_json(directory/'manifest.json',manifest)
        reload_run(directory)
        manifest.update(status='completed',stage='complete',real_weights_executed=real,cuda_executed=real)
        write_json(directory/'manifest.json',manifest)
        print('完成：',directory)
        print('同一量化基座测试loss：',before_loss['loss'],'→',after_loss['loss'])
        print('严格成功：',before_eval['strict_success_rate'],'→',after_eval['strict_success_rate'])
        if not real:print('仅教学随机基座＋纯PyTorch NF4码本；不是bitsandbytes/Qwen/CUDA成绩。')
        return directory
    except Exception as exc:
        manifest.update(status='failed',error=f'{type(exc).__name__}: {exc}')
        write_json(directory/'manifest.json',manifest)
        (directory/'error.txt').write_text(traceback.format_exc(),encoding='utf-8');raise

def rescore_run(directory):
    directory=Path(directory)
    parts={k:read_jsonl(directory/f'{k}_messages.jsonl') for k in ('train','dev','test')}
    contract=json.loads((directory/'contract.json').read_text(encoding='utf-8'))
    require(digest(parts)==contract['data_sha256'],'数据指纹变化')
    def read(path):
        with open(path,encoding='utf-8') as f:return [json.loads(line) for line in f if line.strip()]
    base=aggregate(parts['test'],read(directory/'base_raw.jsonl'))
    after=aggregate(parts['test'],read(directory/'adapter_raw.jsonl'))
    report=json.loads((directory/'report.json').read_text(encoding='utf-8'))
    require(base==report['base']['generation'] and after==report['adapter']['generation'],'评分结果不符')
    result={'base':base,'adapter':after,'paired':paired(base,after)}
    write_json(directory/'rescored.json',result);print('重评分一致：',directory);return result

### 全部固定检查与练习检查器

下面折叠单元包含固定检查与参考实现定义。先运行它，然后运行检查。练习区在后文，参考实现用`answer_*`命名，不会覆盖你的`*_ex`函数。

固定检查只验证离线数学/数据/流程，不冒充真实bitsandbytes/PEFT接口测试。

In [19]:
# %% 09. 独立练习与自动检查。答案函数与学生函数不重名。
def answer_a(x,weight_hat,a,b,alpha):
    require(x.shape[-1]==weight_hat.shape[1] and a.shape[1]==x.shape[-1] and b.shape==(weight_hat.shape[0],a.shape[0]),'形状不符')
    require(a.shape[0]>0 and math.isfinite(alpha) and alpha>0,'rank/alpha无效')
    return F.linear(x,weight_hat)+(alpha/a.shape[0])*F.linear(F.linear(x,a),b)

def check_exercise_a(fn):
    torch.manual_seed(363)
    x=torch.randn(2,3,7,dtype=torch.float64,requires_grad=True)
    w=torch.randn(5,7,dtype=torch.float64)
    a=torch.randn(2,7,dtype=torch.float64,requires_grad=True)
    b=torch.randn(5,2,dtype=torch.float64,requires_grad=True)
    expected=answer_a(x,w,a,b,4);actual=fn(x,w,a,b,4)
    torch.testing.assert_close(actual,expected)
    upstream=torch.randn_like(actual)
    eg=torch.autograd.grad(expected,(x,a,b),upstream,retain_graph=True)
    ag=torch.autograd.grad(actual,(x,a,b),upstream)
    for left,right in zip(ag,eg):torch.testing.assert_close(left,right)
    with torch.no_grad():b.zero_()
    torch.testing.assert_close(fn(x,w,a,b,4),F.linear(x,w))
    return 'A通过：三维输入、缩放、输入与A/B梯度、初始等价'

def answer_b(model,x,target,optimizer):
    return adapter_only_update(model,x,target,optimizer)

def check_exercise_b(fn):
    torch.manual_seed(364)
    layer=PackedNF4Linear(nn.Linear(7,5,bias=False),8)
    model=ToyLoRALinear(layer,2,4)
    before=frozen_state_hash(model);old=model.lora_B.detach().clone()
    optimizer=torch.optim.SGD([p for _,p in adapter_parameters(model)],lr=0.01)
    x=torch.randn(4,7);target=torch.randn(4,5)
    # 故意遗留旧梯度，正确实现需清理。
    for _,p in adapter_parameters(model):p.grad=torch.ones_like(p)*1000
    reference=copy.deepcopy(model);ref_opt=torch.optim.SGD([p for _,p in adapter_parameters(reference)],lr=0.01)
    answer_b(reference,x,target,ref_opt);fn(model,x,target,optimizer)
    for (n,p),(rn,rp) in zip(adapter_parameters(model),adapter_parameters(reference)):torch.testing.assert_close(p,rp)
    require(frozen_state_hash(model)==before,'冻结码本/尺度被改写')
    require(not torch.equal(model.lora_B,old),'B没有更新')
    return 'B通过：清梯度、一次更新、精确优化器集合、固定量化基座'

def answer_c(raw_base_weight,weight_hat,a,b,alpha):
    require(raw_base_weight.shape==weight_hat.shape,'基座形状不符')
    delta=(alpha/a.shape[0])*(b@a)
    correct=weight_hat+delta
    wrong=raw_base_weight+delta
    return {'effective_weight':correct,'wrong_base_weight':wrong,
            'base_mismatch':wrong-correct}

def check_exercise_c(fn):
    torch.manual_seed(365);w=torch.randn(5,7)
    q=dequantize_nf4_teaching(quantize_nf4_teaching(w,8));a=torch.randn(2,7);b=torch.randn(5,2)
    copies=[t.clone() for t in (w,q,a,b)]
    got=fn(w,q,a,b,6);expected=answer_c(w,q,a,b,6)
    for key in expected:torch.testing.assert_close(got[key],expected[key])
    for new,old in zip((w,q,a,b),copies):torch.testing.assert_close(new,old,atol=0,rtol=0)
    torch.testing.assert_close(got['base_mismatch'],w-q,atol=1e-6,rtol=1e-5)
    return 'C通过：量化基座合并公式、原浮点基座错配、源张量不变'

class CheckBook:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def close(self,name,a,b,**kwargs):
        torch.testing.assert_close(a,b,**kwargs);self.names.append(name)
    def raises(self,name,fn,exc=(ValueError,TypeError)):
        try:fn()
        except exc:self.names.append(name);return
        raise AssertionError(name+'：未拒绝错误输入')

def run_checks(verbose=True):
    torch.set_num_threads(1);torch.manual_seed(366);book=CheckBook()
    code=torch.arange(16,dtype=torch.uint8)
    book.ok('码本有16项',len(NF4_VALUES)==16)
    book.ok('码本有精确零',NF4_VALUES[7]==0)
    book.ok('码本严格递增',all(a<b for a,b in zip(NF4_VALUES,NF4_VALUES[1:])))
    book.close('码本不是均匀INT4',torch.tensor(NF4_VALUES)[0],torch.tensor(-1.))
    book.ok('两码打包一字节',pack_codes(code).numel()==8)
    book.ok('低位优先布局',pack_codes(torch.tensor([1,2],dtype=torch.uint8)).item()==33)
    book.close('全部码往返',unpack_codes(pack_codes(code),16),code)
    book.close('奇数码往返',unpack_codes(pack_codes(code[:5]),5),code[:5])
    book.raises('拒绝超四位码',lambda:pack_codes(torch.tensor([16],dtype=torch.uint8)))
    book.raises('拒绝浮点码',lambda:pack_codes(torch.ones(3)))
    book.raises('拒绝不匹配载荷',lambda:unpack_codes(pack_codes(code),5))
    w=torch.randn(5,7);p=quantize_nf4_teaching(w,8);q=dequantize_nf4_teaching(p)
    book.ok('恢复原矩阵形状',q.shape==w.shape)
    book.ok('分块补齐',p['padded_count']==40)
    book.ok('实际码载荷20字节',p['packed'].numel()*p['packed'].element_size()==20)
    book.ok('5个尺度',p['scales'].numel()==5)
    book.ok('有限反量化',torch.isfinite(q).all())
    book.ok('原权重通常不等于反量化权重',not torch.equal(w,q))
    z=dequantize_nf4_teaching(quantize_nf4_teaching(torch.zeros(3,5),8))
    book.close('全零组精确恢复',z,torch.zeros(3,5))
    book.raises('拒绝零块长',lambda:quantize_nf4_teaching(w,0))
    book.raises('拒绝NaN',lambda:quantize_nf4_teaching(torch.tensor([[float('nan')]])))
    linear=nn.Linear(7,5);frozen=PackedNF4Linear(linear,8)
    x=torch.randn(2,3,7,requires_grad=True);y=frozen(x)
    book.close('FrozenLinear前向',y,F.linear(x,frozen.dequantized_weight(),frozen.bias))
    y.sum().backward()
    book.ok('冻结层保留输入梯度',x.grad is not None and x.grad.norm()>0)
    book.ok('无可训练整数码或尺度',len(list(frozen.parameters()))==0)
    book.ok('码尺度是buffer',{'packed','scales','codebook','bias'}<=dict(frozen.named_buffers()).keys())
    copied=PackedNF4Linear(nn.Linear(7,5),8);copied.load_state_dict(frozen.state_dict(),strict=True)
    book.close('打包state_dict重载',copied(x),frozen(x),atol=0,rtol=0)
    model=ToyLoRALinear(frozen,2,4);initial=model(x)
    book.close('初始化匹配量化基座',initial,frozen(x),atol=0,rtol=0)
    book.ok('基座与adapter形状不同',model.lora_A.shape==(2,7) and model.lora_B.shape==(5,2))
    initial.sum().backward()
    book.ok('A首步梯度是零不是None',model.lora_A.grad is not None and not torch.count_nonzero(model.lora_A.grad))
    book.ok('B首步非零',model.lora_B.grad is not None and model.lora_B.grad.norm()>0)
    old=frozen_state_hash(model)
    optimizer=torch.optim.Adam([p for _,p in adapter_parameters(model)],lr=.01)
    _,norms=adapter_only_update(model,torch.randn(4,7),torch.randn(4,5),optimizer)
    book.ok('训练后量化状态不变',frozen_state_hash(model)==old)
    book.ok('FP32 adapter',all(p.dtype==torch.float32 for _,p in adapter_parameters(model)))
    expected=F.linear(x,frozen.dequantized_weight(),frozen.bias)+(4/2)*F.linear(F.linear(x,model.lora_A),model.lora_B)
    book.close('非零增量保持alpha/r',model(x),expected)
    saved_before=frozen_state_hash(model)
    with torch.no_grad():frozen.scales[0].add_(0.01)
    book.ok('哈希能发现仅尺度变化',frozen_state_hash(model)!=saved_before)
    with torch.no_grad():frozen.scales[0].sub_(0.01)
    scaled_after=frozen_state_hash(model)
    with torch.no_grad():model.lora_A.add_(0.001)
    book.ok('哈希排除合法adapter更新',frozen_state_hash(model)==scaled_after)

    wrongly_frozen=copy.deepcopy(model);wrongly_frozen.requires_grad_(False)
    book.raises('检查出错序prepare冻结adapter',lambda:adapter_parameters(wrongly_frozen))
    wrong_optimizer=torch.optim.Adam([nn.Parameter(torch.randn(2,2))])
    book.raises('拒绝旧优化器',lambda:assert_optimizer(model,wrong_optimizer))
    with tempfile.TemporaryDirectory() as tmp:
        path=Path(tmp)/'state.pt';torch.save(model.state_dict(),path)
        other=copy.deepcopy(model);other.load_state_dict(torch.load(path,weights_only=True),strict=True)
        book.close('磁盘往返',model(x),other(x),atol=0,rtol=0)
    p2=copy.deepcopy(p);p2['scales'][0]+=1
    book.ok('尺度变化影响反量化',not torch.equal(q,dequantize_nf4_teaching(p2)))
    book.ok('源码状态哈希包含尺度',frozen_state_hash(frozen)!=frozen_state_hash(PackedNF4Linear(nn.Linear(7,5),8)))
    a=torch.randn(2,7);b=torch.randn(5,2)
    merged=q+(b@a)
    book.close('正确合并等于分支和',F.linear(x,q)+F.linear(F.linear(x,a),b),F.linear(x,merged),atol=2e-6,rtol=1e-5)
    book.ok('合并到原基座不等价',not torch.allclose(w+b@a,q+b@a))
    rec=make_records();parts=split_three(rec)
    book.ok('来源分割16/4/4',[v['groups'] for v in validate_splits(parts).values()]==[16,4,4])
    book.ok('来源隔离',not ({r['group_id'] for r in parts['train']}&{r['group_id'] for r in parts['test']}))
    codec=ToyChatCodec();rows=prepare_splits(parts,codec,256)
    book.ok('同位标签',all(y==IGNORE or y==i for r in rows['train'] for i,y in zip(r['input_ids'],r['labels'])))
    book.ok('只监督末轮',sum(y!=IGNORE for y in rows['train'][0]['labels'])==len(parts['train'][0]['messages'][-1]['content'].encode())+1)
    batch=tensor_batch(rows['train'][:2],codec.pad_id,torch.device('cpu'))
    book.ok('标签long',batch['labels'].dtype==torch.long)
    book.ok('有效结束符没删除',bool((batch['labels']==codec.eot_id).any()))
    test_logits=torch.randn(*batch['labels'].shape,260,requires_grad=True)
    summed,count=loss_parts(test_logits,batch['labels']);summed.backward()
    book.ok('loss可微',test_logits.grad is not None and bool(torch.isfinite(test_logits.grad).all()))
    book.ok('正确统计移位目标',count==int((batch['labels'][:,1:]!=IGNORE).sum()))
    invalid=torch.full_like(batch['labels'],IGNORE)
    book.raises('全忽略拒绝',lambda:loss_parts(test_logits,invalid))
    toy=make_toy_model(Settings());book.ok('教学量化内部12个linear',payload_report(toy)['packed_nf4_modules']==12)
    book.ok('只有A/B可训练',all('lora_' in n for n,p in toy.named_parameters() if p.requires_grad))
    book.ok('Q/V参数2048',sum(p.numel() for _,p in adapter_parameters(toy))==2048)
    head=toy.base.head;book.ok('输出头保持浮点',isinstance(head,nn.Linear) and head.weight.dtype==torch.float32)
    sel=DevSelector();book.raises('拒绝test选轮',lambda:sel.consider('test',1,1,toy))
    sel.consider('dev',0,2,toy);snapshot=copy.deepcopy(sel.state)
    with torch.no_grad():next(p for _,p in adapter_parameters(toy)).add_(1)
    book.ok('最佳快照独立',all(torch.equal(sel.state[k],v) for k,v in snapshot.items()))
    book.ok('并列保留早轮',not sel.consider('dev',1,2,toy))
    result=aggregate(parts['test'],[]);book.ok('缺失保留分母',result['planned']==8 and result['strict_success_rate']==0)
    good=parts['test'][0]
    result=aggregate(parts['test'],[{'id':good['id'],'text':good['messages'][-1]['content'],'error':None,'stop_reason':'budget'}])
    book.ok('预算结束不算严格成功',result['answer_correct_rate']==1/8 and result['strict_success_rate']==0)
    book.raises('JSON重复键拒绝',lambda:quantity_object('{"quantity":1,"quantity":2}'))
    book.raises('JSON布尔拒绝',lambda:quantity_object('{"quantity":true}'))
    spec=quantization_spec(Settings(device='cuda',dtype='float16'))
    book.ok('NF4双重量化计划',spec['bnb_4bit_quant_type']=='nf4' and spec['bnb_4bit_use_double_quant'])
    book.ok('FP16计算uint8载荷',spec['bnb_4bit_compute_dtype']=='float16' and spec['bnb_4bit_quant_storage']=='uint8')
    book.ok('不自动CPU卸载',not spec['llm_int8_enable_fp32_cpu_offload'])
    book.raises('真实量化不接受FP32设置',lambda:quantization_spec(Settings()))
    cfg=SimpleNamespace(hidden_size=1024,num_attention_heads=16,num_key_value_heads=8,head_dim=128,
                        intermediate_size=3072,num_hidden_layers=28)
    base=SimpleNamespace(config=cfg);plan=target_plan(base,Settings())
    book.ok('真实Q/V有56层',len(plan)==56)
    book.ok('显式head_dim',plan['model.layers.0.self_attn.q_proj']==(1024,2048))
    book.ok('r8 Q/V预算',sum(8*(i+o) for i,o in plan.values())==1146880)
    all_plan=target_plan(base,Settings(targets='all-linear'))
    book.ok('7投影计划196层',len(all_plan)==196)
    book.ok('all-linear不含词表头',all('lm_head' not in n for n in all_plan))
    book.ok('all-linear参数预算5046272',sum(8*(i+o) for i,o in all_plan.values())==5046272)
    book.ok('无GPU不伪造显存',cuda_memory(torch.device('cpu'))['allocated'] is None)
    book.ok('练习A参考答案',bool(check_exercise_a(answer_a)))
    book.ok('练习B参考答案',bool(check_exercise_b(answer_b)))
    book.ok('练习C参考答案',bool(check_exercise_c(answer_c)))
    if verbose:print(f'离线固定检查通过：{len(book.names)} 项；不包含bitsandbytes/PEFT/CUDA真实执行。')
    return {'passed':len(book.names),'checks':book.names,'real_bnb_executed':False,'real_peft_executed':False,'cuda_executed':False}

In [20]:
check_report = run_checks()
print("真实bnb执行：", check_report["real_bnb_executed"])
print("CUDA执行：", check_report["cuda_executed"])

离线固定检查通过：75 项；不包含bitsandbytes/PEFT/CUDA真实执行。
真实bnb执行： False
CUDA执行： False


## 10. 实际离线SFT实验

这次把教学网络内部的12个线性层换成打包NF4模块，Embedding、位置表、归一化和输出头保持FP32；只在Q/V投影训练手写LoRA。网络是随机基座，不是预训练Qwen。

每次前向会反量化整层，所以下面的训练只检验连接和数值，不拿耗时或内存当作QLoRA后端的表现。

默认运行8轮，微批次2、累积4，实际32次参数更新。可关闭这个开关先学数学，但已有输出是实际CPU运行，不是预填示例。

In [21]:
RUN_OFFLINE_TRAINING = True
OFFLINE_SETTINGS = Settings(output_dir="lesson_36_outputs", epochs=8,
                            gradient_checkpointing=True)
if RUN_OFFLINE_TRAINING:
    offline_run = run_pipeline(OFFLINE_SETTINGS, real=False)
else:
    offline_run = None
    print("离线训练未运行。")

epoch 01 train=5.47055 dev=5.39767 selected=True


epoch 02 train=5.23329 dev=5.17806 selected=True


epoch 03 train=5.00489 dev=4.96967 selected=True


epoch 04 train=4.87683 dev=4.85019 selected=True


epoch 05 train=4.73423 dev=4.71321 selected=True


epoch 06 train=4.60484 dev=4.58917 selected=True


epoch 07 train=4.51334 dev=4.49560 selected=True


epoch 08 train=4.43339 dev=4.41592 selected=True


相同量化基座重载： 0.0 ；生成相同： True
完成： /mnt/data/_lesson36_build/notebook_run/lesson_36_outputs/offline_20261102T051732_987310Z
同一量化基座测试loss： 5.566801897684733 → 4.44112720489502
严格成功： 0.0 → 0.0
仅教学随机基座＋纯PyTorch NF4码本；不是bitsandbytes/Qwen/CUDA成绩。


In [22]:
if offline_run is not None:
    result = json.loads((offline_run / "report.json").read_text(encoding="utf-8"))
    audit = json.loads((offline_run / "audit.json").read_text(encoding="utf-8"))
    print("选择epoch：", result["selected_epoch"])
    print("测试交叉熵：", result["base"]["teacher_forced"]["loss"], "→", result["adapter"]["teacher_forced"]["loss"])
    print("严格成功率：", result["base"]["generation"]["strict_success_rate"], "→", result["adapter"]["generation"]["strict_success_rate"])
    print("基座含量化状态保持不变：", audit["frozen_state_unchanged"])
    print("再次提醒：不是预训练模型能力或真实低比特性能数据。")

选择epoch： 8
测试交叉熵： 5.566801897684733 → 4.44112720489502
严格成功率： 0.0 → 0.0
基座含量化状态保持不变： True
再次提醒：不是预训练模型能力或真实低比特性能数据。


loss下降不意味着严格任务成功。随机冻结基座上有限的adapter训练，可能提升参考答案的平均概率，却仍然无法自由生成符合格式的答案。不要删除失败样本，也不要把较低交叉熵改名为“正确率提高”。

## 11. 保存与重载：adapter必须知道它接在哪个基座上

训练时的有效权重为 $\widehat W_0+\Delta W$。加载同一adapter到原浮点基座，得到 $W_0+\Delta W$，两者通常不同。重新量化合并后的权重也可能引入另一轮舍入误差。

本课因此**不做权重合并**：先按同一revision和量化配置重建基座，再加载adapter，核对开发集探针的logits和贪心编号。[8]

真实adapter文件不包含完整基座；本课输出也不是第26课的精确断点续训检查点。加载前检查软件版本、量化配置、模板和量化状态指纹，不假定跨环境逐位复现。

In [23]:
for key in ["wrong_base_merge_weight_max_diff", "requantize_merged_weight_max_diff"]:
    print(key, ":", mechanisms[key])
if offline_run is not None:
    loaded_report = reload_run(offline_run)
    print("推理重载可训练参数：", loaded_report["trainable_parameters"])
    rescored = rescore_run(offline_run)
    print("重新评分计划题数：", rescored["base"]["planned"])

wrong_base_merge_weight_max_diff : 0.0359647274017334
requantize_merged_weight_max_diff : 0.03998847305774689
相同量化基座重载： 0.0 ；生成相同： True
推理重载可训练参数： 0
重评分一致： /mnt/data/_lesson36_build/notebook_run/lesson_36_outputs/offline_20261102T051732_987310Z
重新评分计划题数： 8


## 12. 在你的3060上运行真实QLoRA

真实路径显式要求CUDA，不会失败后改用教学网络或另一份模型。先使用小批次1、长度256、rank8、梯度累积4和非重入激活检查点；是否能运行由你的实际显存和环境决定。

课程固定依赖（不是最新版本声明；本制作环境没有安装/执行这些真实路径）：

```bash
python -m pip install "transformers==4.57.3" "peft==0.18.0" "accelerate==1.12.0" "bitsandbytes==0.48.2" safetensors
```

保留已有能调用CUDA的PyTorch，不要为此盲目重装驱动。固定版本的bitsandbytes还要求满足系统、CUDA和硬件条件；先查看`--doctor`与安装说明。[9]

先`--inspect`只加载分词器；通过后才`--run`。真实模型默认Qwen3-0.6B固定commit、NF4+双重量化、FP16计算、FP32 adapter、普通AdamW。没有使用分页优化器。

In [24]:
RUN_REAL_INSPECTION = False
RUN_REAL_TRAINING = False
REAL_SETTINGS = Settings(
    output_dir="lesson_36_outputs", device="cuda", dtype="float16",
    epochs=3, micro_batch=1, accumulation=4, lr=2e-4, rank=8, alpha=16,
    gradient_checkpointing=True, targets="qv", max_length=256,
    local_files_only=False,
)
if RUN_REAL_INSPECTION:
    inspect_pipeline(REAL_SETTINGS)
else:
    print("真实分词器检查未开启；没有下载。")
if RUN_REAL_TRAINING:
    real_run = run_pipeline(REAL_SETTINGS, real=True)
else:
    print("真实QLoRA未开启；没有Qwen、PEFT或CUDA结果。")

真实分词器检查未开启；没有下载。
真实QLoRA未开启；没有Qwen、PEFT或CUDA结果。


### 命令行入口

```bash
python lesson_36_qlora_quantized_base_training.py --doctor
python lesson_36_qlora_quantized_base_training.py --checks-only
python lesson_36_qlora_quantized_base_training.py --math-only
python lesson_36_qlora_quantized_base_training.py
```

真实运行：

```bash
python lesson_36_qlora_quantized_base_training.py --inspect
python lesson_36_qlora_quantized_base_training.py --run --device cuda --dtype float16
```

可选：`--targets all-linear`覆盖七类投影；`--dtype bfloat16`须通过硬件检测；本地权重齐全时添加`--local-files-only`。

每次建立新时间戳目录；记录`manifest.json`、`audit.json`、`memory.json`、`contract.json`、`selection.json`、`report.json`、原始回答和adapter。真实失败会记录错误，不会伪造成功文件。

重载/重新评分：

```bash
python lesson_36_qlora_quantized_base_training.py --reload-run 你的实际运行目录
python lesson_36_qlora_quantized_base_training.py --score 你的实际运行目录
```

报告含明文输入输出，分享前去除私人数据。不要把教学adapter传给PEFT，格式不同。

## 13. 内存与实验比较的边界

QLoRA主要减少冻结基座权重载荷，但仍要容纳：量化元数据、未量化权重、A/B、A/B梯度、优化器状态、激活、临时工作区和分配器缓存。`prepare`把某些权重升FP32后，常驻占用甚至可能上升；这不等于量化失败。[4]

显存不足时先降低序列长度或micro-batch，使用累积和激活检查点，再考虑rank与目标范围。它们改变的因素不同，要另开实验记录。不要用“模型能推理”推出“同样长度能训练”。

需要与第35课FP16 LoRA比较时，应在当前环境**重跑匹配基线**：相同模型commit、数据/模板、targets、rank、更新预算、生成设置与统计口径。尤其不要把qv与all-linear的变化都归因于量化。当前默认报告只比较同一NF4基座关闭与开启adapter。

## 14. 独立练习 A：手写固定量化权重上的LoRA前向

输入`x`支持三维；`weight_hat`是不需要梯度的重建权重；A/B需要梯度。实现正确的`alpha/r`，保持输入梯度，返回输出，不修改任何输入。先写完再打开检查。

In [25]:
def qlora_forward_ex(x, weight_hat, a, b, alpha):
    raise NotImplementedError("请实现 F.linear(x, weight_hat) + alpha/r 的低秩分支")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(check_exercise_a(qlora_forward_ex))

## 15. 独立练习 B：仅更新adapter一次

实现清零旧梯度、前向、均方损失、反向与更新；优化器须精确持有A/B，量化码与尺度不变。检查器会故意预留旧梯度，不能忘记`zero_grad()`。

In [26]:
def adapter_step_ex(model, x, target, optimizer):
    raise NotImplementedError("请完成一次仅adapter的更新")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(check_exercise_b(adapter_step_ex))

## 16. 独立练习 C：区分两种“合并基座”

计算正确有效权重`weight_hat + alpha/r * B@A`，以及误用原浮点基座时的权重。返回字典`effective_weight`、`wrong_base_weight`、`base_mismatch`。不得原地修改输入。

为什么差值不依赖A/B？解释这件事，再运行检查。

In [27]:
def merge_equation_ex(raw_base_weight, weight_hat, a, b, alpha):
    raise NotImplementedError("请计算两种有效权重及差值")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(check_exercise_c(merge_equation_ex))

## 17. 参考答案——完成练习后再展开

下列实现与脚本参考答案一致，不覆盖学生函数。检查通过表示符合这里的有限测试，不保证任意工程场景。

In [28]:
def answer_a(x,weight_hat,a,b,alpha):
    require(x.shape[-1]==weight_hat.shape[1] and a.shape[1]==x.shape[-1] and b.shape==(weight_hat.shape[0],a.shape[0]),'形状不符')
    require(a.shape[0]>0 and math.isfinite(alpha) and alpha>0,'rank/alpha无效')
    return F.linear(x,weight_hat)+(alpha/a.shape[0])*F.linear(F.linear(x,a),b)

print(check_exercise_a(answer_a))

A通过：三维输入、缩放、输入与A/B梯度、初始等价


In [29]:
def answer_b(model,x,target,optimizer):
    return adapter_only_update(model,x,target,optimizer)

print(check_exercise_b(answer_b))

B通过：清梯度、一次更新、精确优化器集合、固定量化基座


In [30]:
def answer_c(raw_base_weight,weight_hat,a,b,alpha):
    require(raw_base_weight.shape==weight_hat.shape,'基座形状不符')
    delta=(alpha/a.shape[0])*(b@a)
    correct=weight_hat+delta
    wrong=raw_base_weight+delta
    return {'effective_weight':correct,'wrong_base_weight':wrong,
            'base_mismatch':wrong-correct}

print(check_exercise_c(answer_c))

C通过：量化基座合并公式、原浮点基座错配、源张量不变


## 18. 验收清单

能解释冻结码、可微输入、可训练adapter三个边界；能区分原浮点基座与反量化基座；能说明prepare/inject/optimizer顺序；能从日志确认量化模块和训练参数；能让保存后的adapter在同量化基座上重载一致。

完成真实CUDA实验时，分别确认**加载成功、实际参数更新、任务指标**，不把它们当成同一件事。当前制作环境仅验证CPU路径，GPU显存字段为`null`，不是0。

下一课：**第37课——微调诊断与泛化评测：过拟合、数据泄漏和能力退化**。在已有训练流程上，判断值得保留的是哪一种改动，而不是只追求训练loss更低。

## 参考来源（原始论文、固定版本官方文档与源码）

[1] Dettmers等，QLoRA论文：https://arxiv.org/abs/2305.14314

[2] Transformers 4.57.3 bitsandbytes指南：https://huggingface.co/docs/transformers/v4.57.3/en/quantization/bitsandbytes

[3] bitsandbytes 0.48.2 NF4码本与双重量化实现：https://github.com/bitsandbytes-foundation/bitsandbytes/blob/0.48.2/bitsandbytes/functional.py

[4] PEFT 0.18.0 prepare_model_for_kbit_training源码：https://github.com/huggingface/peft/blob/v0.18.0/src/peft/utils/other.py

[5] bitsandbytes Linear4bit源码：https://github.com/bitsandbytes-foundation/bitsandbytes/blob/0.48.2/bitsandbytes/nn/modules.py

[6] PEFT量化与LoRA指南：https://huggingface.co/docs/peft/v0.18.0/en/developer_guides/quantization

[7] PyTorch CUDA内存统计：https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html

[8] PEFT检查点格式：https://huggingface.co/docs/peft/v0.18.0/en/developer_guides/checkpoint

[9] bitsandbytes固定版本安装：https://huggingface.co/docs/bitsandbytes/v0.48.0/installation

固定Qwen配置：https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/config.json

版本为课程复现实验所固定，不代表最新版本或已验证的CUDA兼容矩阵。